# Multi-Family Empirical Law Discovery v3: LoRA Training Fix

Run all cells in Colab with a GPU. Five relationship families are modeled separately, each with its own price or historical-state baseline. Holm correction is then applied jointly across families.

- `excess_return`: future excess log return, in basis points (bps).
- `realized_risk`: root mean square of future daily log returns, in bps/day. This is RMS, not a demeaned standard deviation.
- `volume_activity`: average future volume z-score. This is not raw trading-volume growth.
- `market_shock`: absolute value of the equal-weight mean future stock log return within the sampled universe, in bps. This is not an exchange-index return.
- `dispersion`: cross-sectional standard deviation of future cumulative stock log returns, in bps.

Each equation names its left-hand-side statistic and time window. Nonlinear terms, interactions, and lagged terms can express state-dependent and dynamic relationships. These are testable statistical models, not identities or established causal laws.

All families' structures are frozen before the final TEST evaluation. Baseline comparisons and news-ablation tests are corrected together across all families. The default historical test period has already been inspected, so findings are exploratory. Changing the outcome does not create independent confirmation.

The default is at most two equations per family. The language-model backbone is loaded once; expression caching and small closed-form matrix solves reduce computation. SFT uses CV-positive diverse structures when available. Otherwise it learns from the best available TRAIN-CV structures as explicitly labeled weak supervision. Negative examples are required only for DPO. Final law-validation thresholds are unchanged. Set `USE_LLM=False` for faster template and random-structure search. Set `DEMO_MODE=True` for a synthetic-price demonstration.

After running, inspect `LAWS.csv`, `LAWS.json`, `LAW_REPORT.md`, and `ALL_PREREGISTERED.json` in the output root. Family-specific subdirectories contain search, training, coefficient, and model records. Each result includes its units, sampling level, and outcome definition. This English edition uses a separate default run folder, `multi_laws_v3_lora_en`.

Validation completed for the underlying version: syntax checks and an end-to-end synthetic-data run without the LLM. This revision changes training-target eligibility and diagnostics. Numerical fitting and final evidence tests retain their existing logic. Real-data execution and GPU training still need to be run in Colab.

Training audit: inspect `training_pool_diagnostics.json`, `training_status.json`, and `train_ei_robust.csv` in each family directory. GPU optimizer execution has not been tested in this environment. A successful update does not guarantee predictive improvement. The previously inspected historical test period remains exploratory.


In [ ]:
# ---------- Paths (case sensitive on Colab) ----------
DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_DATA_PATH = f"{DRIVE_ROOT}/llm_hackathon/data/dataset.parquet"
NEWS_FACTORS_PATH = f"{DRIVE_ROOT}/LLM_hackathon/market_kepler/news_factors/news_factors_trading.csv"
SECTOR_CACHE_INPUT = f"{DRIVE_ROOT}/llm_hackathon/results/alpha_agent_evo/ticker_sectors.csv"
OUTPUT_ROOT = f"{DRIVE_ROOT}/llm_hackathon/results/equation_discovery"
RUN_NAME = "multi_laws_v3_lora_en"
DEMO_MODE = False
FAST = True
SEED = 42
RESUME = True
STUDY_START, VAL_START, TEST_START, STUDY_END = "2015-04-01", "2021-01-01", "2023-01-01", None
HORIZON, EXECUTION_LAG, NEWS_LAG = 5, 1, 1
NEWS_PREFIX = "world_"
MAX_TICKERS = 150 if FAST else None
MIN_NAMES = 20
ALLOW_YFINANCE_LOOKUP = False
HOLDOUT_IS_FRESH = False  # True only for an actually uninspected outcome period

# ---------- Equation structure / coefficient fitting ----------
MAX_TERMS = 6
RIDGE_LAMBDA = 0.001      # loss = date-equal mean squared error + lambda*sum(standardized slopes**2)
CV_FOLDS = 3
CV_MIN_TRAIN_DAYS = 120
MIN_TERM_COVERAGE = 0.80  # check TRAIN only; missing terms use train median, not row deletion
ROBUST_LAMBDA = 0.5
COMPLEXITY_PENALTY = 0.0002
PRICE_BASELINE_FEATURES = ["mom_5", "mom_20", "vol_20", "beta_60", "volume_z_20"]
EQUIVALENCE_RHO = 0.995

# ---------- LLM budgets ----------
USE_LLM = True
HF_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
BASE_ROUNDS = 1 if FAST else 5
POST_ROUNDS = 1 if FAST else 4
PROPOSALS_PER_ROUND = 6
MAX_NEW_TOKENS = 1100
TEMPERATURE = 0.8
RUN_BASE_CONTROL = True
RANDOM_CANDIDATES = 12 if FAST else 48
OBJECTIVES = ["ei_robust"] # supported: ei_mse, ei_robust, dpo, grpo
TRAIN_PROMPTS = 16 if FAST else 48
PROMPTS_PER_STEP = 2
TRAIN_EPOCHS = 1
TARGETS_PER_ANSWER = 2
TRAIN_MAX_TOKENS = 3072
TRAIN_LR = 1e-4
LORA_R, LORA_ALPHA = 16, 32
MIN_TRAIN_SCORE = 0.0     # do not teach weak structures if no CV-positive targets; explicit skip
DPO_BETA, GRPO_BETA, GRPO_K = 0.1, 0.04, 3
REWARD_SCALE = 0.01

# ---------- Frozen evaluation rules ----------
TOP_FINAL = 2
BOOTSTRAPS = 400 if FAST else 1000
COEF_BOOTSTRAPS = 300 if FAST else 1000
BOOT_BLOCK = max(10, HORIZON + EXECUTION_LAG)
NW_LAGS = BOOT_BLOCK
MIN_TEST_DAYS = 120
MIN_CV_CONSISTENCY = 2/3
MAX_TEST_IMPUTED_SHARE = 0.20
assert 1 <= MAX_TERMS <= 8 and HORIZON >= 1 and EXECUTION_LAG >= 1 and NEWS_LAG >= 1
assert set(OBJECTIVES) <= {"ei_mse", "ei_robust", "dpo", "grpo"}

LAW_FAMILIES = ["excess_return", "realized_risk", "volume_activity", "market_shock", "dispersion"]


In [ ]:
import os, sys, json, time, math, re, ast, gc, glob, hashlib, random, contextlib, io, copy
from pathlib import Path
from collections import Counter
import importlib.util, subprocess
need = {"numpy":"numpy", "pandas":"pandas", "scipy":"scipy", "matplotlib":"matplotlib", "IPython":"ipython"}
if not DEMO_MODE:
    need.update({"pyarrow":"pyarrow", "lxml":"lxml", "requests":"requests"})
if USE_LLM and not DEMO_MODE:
    need.update({"torch":"torch", "transformers":"transformers>=4.51,<5", "accelerate":"accelerate", "bitsandbytes":"bitsandbytes", "peft":"peft"})
missing = [pkg for mod,pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display, Markdown, Math
try:
    import torch
except ImportError:
    torch = None
random.seed(SEED); rng = np.random.default_rng(SEED)
if torch is not None: torch.manual_seed(SEED)
pd.set_option("display.max_colwidth",120)
if not DEMO_MODE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass
else:
    USE_LLM = False
    OUTPUT_ROOT = "/content/equation_demo" if Path("/content").exists() else str(Path.cwd()/"equation_demo")
OUTPUT_DIR = str(Path(OUTPUT_ROOT)/(RUN_NAME + ("_SYNTHETIC" if DEMO_MODE else "")))
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)

def resolve_input(path, filename):
    if Path(path).is_file(): return path
    hits=[]
    for depth in range(4):
        hits.extend(glob.glob(os.path.join(DRIVE_ROOT, *(["*"]*depth),filename)))
    hits=sorted(set(hits))
    if len(hits)==1: return hits[0]
    raise FileNotFoundError(f"Set the exact path for {filename}. Matches: {hits[:8]}")
if not DEMO_MODE:
    data_path=resolve_input(DRIVE_DATA_PATH,"dataset.parquet")
    factors_path=resolve_input(NEWS_FACTORS_PATH,"news_factors_trading.csv")
    print("Prices:",data_path,"\nNews:",factors_path)


In [ ]:
if DEMO_MODE:
    dates=pd.bdate_range("2014-12-01","2025-06-30"); rr=np.random.default_rng(SEED)
    demo_tickers=[f"S{i:02d}" for i in range(36)]
    demo_ret=rr.normal(0,.012,(len(dates),len(demo_tickers)))
    frames=[]
    for j,t in enumerate(demo_tickers):
        p=pd.DataFrame({"date":dates,"ticker":t,"close":100*np.exp(np.cumsum(demo_ret[:,j])),"ret_1d":demo_ret[:,j]})
        for w in [5,20,60]: p[f"mom_{w}"]=pd.Series(demo_ret[:,j]).rolling(w).sum()
        p["vol_20"]=pd.Series(demo_ret[:,j]).rolling(20).std()
        p["volume_z_20"]=rr.normal(size=len(p))
        frames.append(p)
    DEMO_PRICES=pd.concat(frames,ignore_index=True)
    ss=["energy","financials","tech","industrials","consumer","healthcare","materials","utilities","real_estate","defense","airlines_travel","agriculture"]
    sec=pd.DataFrame({"ticker":demo_tickers,"name":demo_tickers,"kepler_sector":[ss[i%12] for i in range(len(demo_tickers))]})
    SECTOR_CACHE_INPUT=str(Path(OUTPUT_DIR)/"demo_sectors.csv");sec.to_csv(SECTOR_CACHE_INPUT,index=False)
    nn=pd.DataFrame({"date":dates})
    for col in ["news_shock","news_mood","news_mood_surprise","news_fear_share","topic_energy_oil_mood","topic_central_bank_rates_mood","topic_war_conflict_mood"]:
        nn[col]=rr.normal(size=len(dates))
    nn["news_fear_share"]=rr.uniform(0,1,len(dates));nn.loc[np.arange(len(nn))%5==0,"news_fear_share"]=0
    for s in ss: nn[f"sector_{s}"]=rr.integers(0,10,len(dates));nn[f"sector_{s}_mood"]=rr.normal(size=len(dates))
    factors_path=str(Path(OUTPUT_DIR)/"demo_news.csv");nn.to_csv(factors_path,index=False)


In [ ]:
df = DEMO_PRICES.copy() if DEMO_MODE else pd.read_parquet(data_path)
df["date"] = pd.to_datetime(df["date"])
if getattr(df["date"].dt, "tz", None) is not None:
    df["date"] = df["date"].dt.tz_localize(None)

if MAX_TICKERS:
    keep = rng.choice(np.sort(df["ticker"].unique()), size=min(MAX_TICKERS, df["ticker"].nunique()), replace=False)
    df = df[df["ticker"].isin(keep)]

study_start = pd.Timestamp(STUDY_START)
study_end = pd.Timestamp(STUDY_END) if STUDY_END else df["date"].max()
warm_start = study_start - pd.Timedelta(days=150)          # ~100 trading days of history for ts_* windows
df = df[(df["date"] >= warm_start) & (df["date"] <= study_end)]
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
assert not df.duplicated(["date", "ticker"]).any(), "duplicate (date,ticker) rows"
assert (pd.to_numeric(df["close"], errors="coerce").dropna() > 0).all(), "close must be positive"
num_cols = df.select_dtypes("float64").columns
df[num_cols] = df[num_cols].astype("float32")

TARGET    = "fwd_ret_excess"
LEAK_COLS = {"fwd_ret", "fwd_ret_excess", "label_up", "split", "fwd1_excess"}   # never allowed in formulas

# ---- 1-day forward excess return (close t -> close t+1), for the day-by-day delay profiles only
lp = np.log(df["close"].astype("float64"))
fwd1 = lp.groupby(df["ticker"]).shift(-1) - lp
df["fwd1_excess"] = (fwd1 - fwd1.groupby(df["date"]).transform("mean")).astype("float32")

# ---- price calendar = the dataset's own trading days
CAL = pd.DatetimeIndex(np.sort(df["date"].unique()))
pos = CAL.get_indexer(df["date"])
val_i, test_i = CAL.searchsorted(pd.Timestamp(VAL_START)), CAL.searchsorted(pd.Timestamp(TEST_START))
df["split"] = np.select(
    [df["date"].to_numpy() < np.datetime64(study_start), pos < val_i - HORIZON - EXECUTION_LAG, pos < val_i,
     pos < test_i - HORIZON - EXECUTION_LAG, pos < test_i],
    ["warmup", "train", "gap", "val", "gap"], default="test")

# ---- beta_60: rolling 60-day beta of each stock to the equal-weight market (past and today only)
assert "ret_1d" in df.columns, "dataset.parquet needs a ret_1d column"
mkt_1d = df.groupby("date")["ret_1d"].transform("mean").astype("float64")
r1 = df["ret_1d"].astype("float64")
tmp = pd.DataFrame({"r": r1, "m": mkt_1d, "rm": r1 * mkt_1d, "mm": mkt_1d * mkt_1d})
roll = tmp.groupby(df["ticker"].to_numpy()).rolling(60, min_periods=40).mean().reset_index(level=0, drop=True)
roll = roll.reindex(df.index)
df["beta_60"] = ((roll["rm"] - roll["r"] * roll["m"]) /
                 (roll["mm"] - roll["m"] ** 2).where(lambda v: v > 1e-12)).clip(-3, 5).astype("float32")
del tmp, roll

PRICE_FEATURES = [c for c in df.columns
                  if c not in LEAK_COLS | {"date", "ticker"} and not c.startswith("csr_")
                  and not c.startswith("news_") and pd.api.types.is_numeric_dtype(df[c])]
SPLIT_MASK = {s: (df["split"] == s).to_numpy() for s in ["train", "val", "test"]}
for s, m in SPLIT_MASK.items():
    d = df.loc[m, "date"]
    print(f"{s:>5}: {m.sum():>9,} rows | {d.min().date()} -> {d.max().date()} | {d.nunique()} days")
print(f"\n{df['ticker'].nunique()} tickers | {len(PRICE_FEATURES)} price features: {PRICE_FEATURES}")
# Recompute labels on the GLOBAL trading calendar: missing ticker-days never compress the horizon.
close_panel = df.pivot(index="date", columns="ticker", values="close").reindex(CAL).astype(float)
log_panel = np.log(close_panel)
fwd_panel = log_panel.shift(-(HORIZON + EXECUTION_LAG)) - log_panel.shift(-EXECUTION_LAG)
excess_panel = fwd_panel.sub(fwd_panel.mean(axis=1), axis=0)
ti = CAL.get_indexer(df.date); ni = close_panel.columns.get_indexer(df.ticker)
df["fwd_ret"] = fwd_panel.to_numpy()[ti,ni].astype("float32")
df[TARGET] = excess_panel.to_numpy()[ti,ni].astype("float32")
# Explicit allow-list prevents an unknown future/label column from entering the DSL.
SAFE_PRICE = ["close", "ret_1d", "ret_1d_excess", "mom_5", "mom_20", "mom_60", "mom_120", "mom_252_21",
              "vol_20", "vol_60", "hl_range", "hl_range_20", "ma_gap_50", "ma_gap_200", "volume_z_20",
              "log_dollar_vol_20", "rsi_14", "beta_60"]
PRICE_FEATURES = [c for c in SAFE_PRICE if c in df and pd.api.types.is_numeric_dtype(df[c])]
for period,mask in SPLIT_MASK.items():
    if df.loc[mask & df[TARGET].notna(), "date"].nunique() < 60:
        raise ValueError(f"{period} has fewer than 60 labelled days; check dates and input history")
print("Target rebuilt; execution lag:", EXECUTION_LAG, "holding horizon:", HORIZON)


In [ ]:
KEPLER_SECTORS = ["energy", "financials", "tech", "industrials", "consumer", "healthcare", "materials",
                  "utilities", "real_estate", "defense", "airlines_travel", "agriculture"]
SECTOR_CACHE = os.path.join(OUTPUT_DIR, "ticker_sectors.csv")


def to_kepler_sector(sector, sub=""):
    s, u = str(sector).lower(), str(sub).lower()
    if "airline" in u or "hotels, resorts" in u or "cruise" in u or "travel" in u:
        return "airlines_travel"
    if "aerospace" in u or "defense" in u:
        return "defense"
    if "agricultur" in u or "fertilizer" in u or "farm products" in u:
        return "agriculture"
    for key, val in [("energy", "energy"), ("financ", "financials"), ("information tech", "tech"),
                     ("technology", "tech"), ("communication", "tech"), ("industrial", "industrials"),
                     ("consumer", "consumer"), ("health", "healthcare"), ("material", "materials"),
                     ("utilit", "utilities"), ("real estate", "real_estate")]:
        if key in s:
            return val
    return None


norm_ticker = lambda t: str(t).upper().strip().replace(".", "-")


def load_sectors(tickers):
    """ticker -> raw sector, sub-industry, company name, Kepler sector. Cached in OUTPUT_DIR."""
    tickers = sorted(set(tickers))
    if os.path.exists(SECTOR_CACHE_INPUT):
        cache_path = SECTOR_CACHE_INPUT
    else:
        cache_path = SECTOR_CACHE
    if os.path.exists(cache_path):
        sec = pd.read_csv(cache_path)
        if "name" in sec.columns and set(tickers) <= set(sec["ticker"]):
            return sec
    rows = {}
    own = [c for c in df.columns if c.lower() in ("sector", "gics_sector")]
    own_name = [c for c in df.columns if c.lower() in ("name", "company", "company_name", "security")]
    if own:
        sub = [c for c in df.columns if c.lower() in ("industry", "sub_industry", "gics_sub_industry")]
        first = df.groupby("ticker")[own + sub + own_name].first()
        for t, r in first.iterrows():
            rows[t] = (r[own[0]], r[sub[0]] if sub else "", "dataset", r[own_name[0]] if own_name else "")
    try:                                           # Wikipedia also gives company names for the company test
        if set(tickers) <= set(rows):
            raise RuntimeError("dataset sector mapping complete; network lookup skipped")
        import requests
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                            headers={"User-Agent": "Mozilla/5.0 (research notebook)"}, timeout=30).text
        wiki = pd.read_html(io.StringIO(html))[0]
        wiki["t"] = wiki["Symbol"].map(norm_ticker)
        wmap = wiki.set_index("t")[["GICS Sector", "GICS Sub-Industry", "Security"]].to_dict("index")
        for t in tickers:
            w = wmap.get(norm_ticker(t))
            if w is None:
                continue
            if t in rows:
                rows[t] = (*rows[t][:3], rows[t][3] or w["Security"])
            else:
                rows[t] = (w["GICS Sector"], w["GICS Sub-Industry"], "wikipedia", w["Security"])
    except Exception as e:
        print("Wikipedia lookup failed:", type(e).__name__, e)
    missing = [t for t in tickers if t not in rows]
    if missing and ALLOW_YFINANCE_LOOKUP:
        print(f"yfinance lookup for {len(missing)} tickers ...")
        import yfinance as yf
        for t in missing:
            try:
                info = yf.Ticker(norm_ticker(t)).info
                rows[t] = (info.get("sector", ""), info.get("industry", ""), "yfinance",
                           info.get("shortName") or info.get("longName") or "")
            except Exception:
                rows[t] = ("", "", "none", "")
            time.sleep(0.2)
    sec = pd.DataFrame([(t, *rows.get(t, ("", "", "none", ""))) for t in tickers],
                       columns=["ticker", "raw_sector", "raw_sub", "source", "name"])
    sec["kepler_sector"] = [to_kepler_sector(a, b) for a, b in zip(sec.raw_sector, sec.raw_sub)]
    sec.to_csv(SECTOR_CACHE, index=False)
    return sec


SECTORS = load_sectors(df["ticker"].unique())
TICKER_SECTOR = SECTORS.set_index("ticker")["kepler_sector"].to_dict()
print(SECTORS["kepler_sector"].value_counts(dropna=False).to_string())
print(f"{SECTORS.kepler_sector.isna().sum()} tickers without a sector get no stock-level news (NaN). "
      f"{(SECTORS['name'].fillna('') != '').sum()} tickers have a company name.")


In [ ]:
raw = pd.read_csv(factors_path, parse_dates=["date"])
print(f"{len(raw):,} rows, {raw.date.min().date()} -> {raw.date.max().date()}, {raw.shape[1] - 1} columns")

TOPICS = ["war_conflict", "trade_tariffs", "energy_oil", "central_bank_rates", "economy_data", "natural_disaster",
          "politics_election", "tech_regulation", "health_pandemic", "corporate_business"]
REGIONS = ["us", "china", "europe", "russia_ukraine", "middle_east", "asia_other", "global", "other"]
EMOTION = ["news_shock", "news_mood", "news_mood_surprise", "news_fear_share"]
GROUP_COLS = [c for c in raw.columns if c.startswith(("topic_", "region_", "sector_"))]
COUNT_COLS = [c for c in ["n_calendar_days", "n_relevant"] if c in raw.columns]

# ---- snap to the price calendar
i = CAL.searchsorted(raw["date"].to_numpy())
raw = raw[i < len(CAL)].copy()
raw["td"] = CAL[i[i < len(CAL)]]
num = raw.drop(columns=["date", "td"]).replace([np.inf, -np.inf], np.nan)
agg = {c: ("sum" if c in COUNT_COLS else "mean") for c in num.columns}
N = num.groupby(raw["td"]).agg(agg).reindex(CAL)
n_merged = int((raw.groupby("td").size() > 1).sum())
covered = (CAL >= raw["td"].min()) & (CAL <= raw["td"].max())

# ---- clip to the train range, fill "no news on this group" with 0
_raw_train = SPLIT_MASK["train"] & np.isfinite(df[TARGET].to_numpy(float))
_counts = np.bincount(CAL.get_indexer(df.loc[_raw_train, "date"]), minlength=len(CAL))
_cap_days = np.flatnonzero(_counts >= MIN_NAMES)
_cap_end = _cap_days[int(.4 * len(_cap_days))] - HORIZON - EXECUTION_LAG
train_cal = covered & CAL.isin(df.loc[SPLIT_MASK["train"], "date"].unique()) & (np.arange(len(CAL)) < _cap_end)
NEWS_CAP_END = str(CAL[_cap_end].date())
NEWS_CLIP_BOUNDS = {}
print("News clipping fitted strictly before", NEWS_CAP_END, "(earliest CV training prefix).")
for c in EMOTION + GROUP_COLS:
    v = N.loc[train_cal, c]
    v = v[v.notna() & (v != 0)]                     # sparse columns: use the non-zero days, or 0 would be the cap
    if len(v) >= 50:
        lo, hi = v.quantile([0.005, 0.995])
        NEWS_CLIP_BOUNDS[c] = [float(min(lo,0)), float(max(hi,0))]
        N[c] = N[c].clip(*NEWS_CLIP_BOUNDS[c])
observed_dates = CAL.isin(raw["td"].unique())
N.loc[observed_dates, GROUP_COLS] = N.loc[observed_dates, GROUP_COLS].fillna(0.0)
N.loc[~observed_dates] = np.nan
N.loc[~covered] = np.nan
N.index.name = "date"
NEWS_DAILY = N.astype("float64")
SECTOR_NEWS = [s for s in KEPLER_SECTORS if f"sector_{s}_mood" in NEWS_DAILY.columns]

print(f"{n_merged} trading days got more than one news row (merged). "
      f"{NEWS_DAILY['news_mood'].notna().sum():,} trading days with news factors.")
display(NEWS_DAILY[EMOTION].describe().T.round(3))
print("Share of trading days with any news on each sector:")
print((NEWS_DAILY.loc[covered, [f"sector_{s}" for s in SECTOR_NEWS]] > 0).mean().round(3).to_string())
P = NEWS_PREFIX
NEWS_LAGGED = NEWS_DAILY.shift(NEWS_LAG)                  # on the trading-day calendar
assert SECTOR_NEWS == KEPLER_SECTORS, f"news file lacks sectors: {set(KEPLER_SECTORS) - set(SECTOR_NEWS)}"

sec_mood = NEWS_LAGGED[[f"sector_{s}_mood" for s in KEPLER_SECTORS]].to_numpy()
sec_vol = NEWS_LAGGED[[f"sector_{s}" for s in KEPLER_SECTORS]].to_numpy()
sm = pd.DataFrame(sec_mood, index=CAL)
sec_surp = (sm - sm.rolling(20, min_periods=10).mean().shift(1)).to_numpy()
sec_decay = sm.ewm(halflife=3, ignore_na=True).mean().where(sm.notna()).to_numpy()

d_pos = CAL.get_indexer(df["date"])
sector_idx = {t: (KEPLER_SECTORS.index(s) if s in KEPLER_SECTORS else -1) for t, s in TICKER_SECTOR.items()}
s_pos = df["ticker"].map(sector_idx).fillna(-1).astype(int).to_numpy()
has = s_pos >= 0


def per_stock(panel):
    out = np.full(len(df), np.nan, dtype="float64")
    out[has] = panel[d_pos[has], s_pos[has]]
    return out.astype("float32")


slog = lambda x: np.sign(x) * np.log1p(np.abs(x))
df[f"{P}sent"] = per_stock(sec_mood)
df[f"{P}count"] = slog(per_stock(sec_vol))
df[f"{P}surprise"] = per_stock(sec_surp)
df[f"{P}decay"] = per_stock(sec_decay)
for name, col in [("mood", "news_mood"), ("mood_surprise", "news_mood_surprise"),
                  ("fear", "news_fear_share"), ("shock", "news_shock"),
                  ("oil", "topic_energy_oil_mood"), ("rates", "topic_central_bank_rates_mood"),
                  ("war", "topic_war_conflict_mood")]:
    df[f"{P}{name}"] = NEWS_LAGGED[col].to_numpy()[d_pos].astype("float32")

NEWS_FEATURES = [f"{P}{n}" for n in ["sent", "count", "surprise", "decay",
                                     "mood", "mood_surprise", "fear", "shock", "oil", "rates", "war"]]
MARKET_WIDE = [f"{P}{n}" for n in ["mood", "mood_surprise", "fear", "shock", "oil", "rates", "war"]]
ALL_FEATURES = PRICE_FEATURES + NEWS_FEATURES

print("Coverage of news features inside the study window:")
inside = df["split"].isin(["train", "val", "test"])
display(df.loc[inside, NEWS_FEATURES].notna().mean().round(3).to_frame("non-null share").T)

FEATURE_DOCS = {
    "close":             "adjusted close price in USD",
    "ret_1d":            "1-day log return",
    "ret_1d_excess":     "1-day log return minus the cross-sectional mean that day",
    "mom_5":             "5-day log return",
    "mom_20":            "20-day log return",
    "mom_60":            "60-day log return",
    "mom_120":           "120-day log return",
    "mom_252_21":        "12-1 month momentum: log return from t-252 to t-21 (skips the last month)",
    "vol_20":            "20-day standard deviation of daily log returns",
    "vol_60":            "60-day standard deviation of daily log returns",
    "hl_range":          "log(high/low) today - intraday range",
    "hl_range_20":       "20-day average of hl_range",
    "ma_gap_50":         "close / 50-day moving average - 1",
    "ma_gap_200":        "close / 200-day moving average - 1",
    "volume_z_20":       "z-score of log volume relative to its own 20-day mean and std",
    "log_dollar_vol_20": "log of 20-day average dollar volume (liquidity / size proxy)",
    "rsi_14":            "14-day relative strength index, 0-100",
    "beta_60":           "60-day beta of the stock's daily return to the equal-weight market return",
    f"{P}sent":          "WORLD NEWS (yesterday) for this stock's sector: signed news intensity, sum(weight x mood) / "
                         "normal news volume. Below 0 = bad world news for the sector, 0 = no sector news",
    f"{P}count":         "WORLD NEWS (yesterday): how much world news hit this stock's sector, log-scaled. 0 = none",
    f"{P}surprise":      f"WORLD NEWS: {P}sent minus its average over the previous 20 trading days",
    f"{P}decay":         f"WORLD NEWS: exponentially decayed {P}sent (half-life 3 days), slow-moving sector mood",
    f"{P}mood":          "WORLD NEWS, MARKET-WIDE (same for every stock on a day): average mood of yesterday's "
                         "important world news, -2 (very bad) to +2 (very good)",
    f"{P}mood_surprise": "WORLD NEWS, MARKET-WIDE: yesterday's mood minus its 30-day average",
    f"{P}fear":          "WORLD NEWS, MARKET-WIDE: share of yesterday's important news that was clearly bad, 0 to 1",
    f"{P}shock":         "WORLD NEWS, MARKET-WIDE: yesterday's news volume vs. the last 90 days (1 = normal, 2 = double)",
    f"{P}oil":           "WORLD NEWS, MARKET-WIDE: signed mood of yesterday's oil / energy news (below 0 = bad)",
    f"{P}rates":         "WORLD NEWS, MARKET-WIDE: signed mood of yesterday's central-bank / interest-rate news",
    f"{P}war":           "WORLD NEWS, MARKET-WIDE: signed mood of yesterday's war / conflict news",
}
for f in ALL_FEATURES:
    FEATURE_DOCS.setdefault(f, "feature from dataset.parquet")
FEATURE_DOC = "\n".join(f"- {f}: {FEATURE_DOCS[f]}" for f in ALL_FEATURES)

FEATURES = list(ALL_FEATURES)
DATE_ARR = df["date"].to_numpy()

# Candidate input vocabulary: no adjusted price level, no rank-only output objective.
FEATURES = [f for f in ALL_FEATURES if f != "close"]
ALL_FEATURES = FEATURES


In [ ]:

# All labels use the GLOBAL trading calendar and windows strictly after the signal.
STOCK_DF=df.copy(); STOCK_FEATURES=list(FEATURES); ROOT_OUTPUT=OUTPUT_DIR
p=STOCK_DF.pivot(index="date",columns="ticker",values="close").reindex(CAL).astype(float)
lp=np.log(p); dr=lp.diff(); fwd=lp.shift(-EXECUTION_LAG-HORIZON)-lp.shift(-EXECUTION_LAG)
def future_mean(a):
    # Arithmetic addition preserves NaN: every day in the label window must be observed.
    parts=[a.shift(-EXECUTION_LAG-j) for j in range(1,HORIZON+1)]
    return sum(parts)/HORIZON
risk=1e4*np.sqrt(future_mean(dr*dr))
volume=STOCK_DF.pivot(index="date",columns="ticker",values="volume_z_20").reindex(index=CAL,columns=p.columns) if "volume_z_20" in STOCK_DF else p*np.nan
activity=future_mean(volume)
# Price-derived baseline features use past data only; correct units are explicit.
STOCK_DF["past_rms_5"]=(1e4*np.sqrt((dr*dr).rolling(5,min_periods=5).mean())).to_numpy()[CAL.get_indexer(STOCK_DF.date),p.columns.get_indexer(STOCK_DF.ticker)]
STOCK_DF["past_rms_20"]=(1e4*np.sqrt((dr*dr).rolling(20,min_periods=20).mean())).to_numpy()[CAL.get_indexer(STOCK_DF.date),p.columns.get_indexer(STOCK_DF.ticker)]
STOCK_FEATURES += ["past_rms_5","past_rms_20"]
FEATURE_DOCS.update(past_rms_5="Past five daily log returns RMS, bps/day",past_rms_20="Past twenty daily log returns RMS, bps/day")
ROW_D=CAL.get_indexer(STOCK_DF.date);ROW_N=p.columns.get_indexer(STOCK_DF.ticker)
market_valid=fwd.notna().sum(axis=1)>=MIN_NAMES
past=lp-lp.shift(HORIZON)
MARKET=STOCK_DF.groupby("date")[STOCK_FEATURES].mean().reindex(CAL)
MARKET["past_market_abs"]=1e4*past.mean(axis=1).abs()
MARKET["past_dispersion"]=1e4*past.std(axis=1,ddof=1)
MARKET["ret_1d"]=dr.mean(axis=1)
MARKET["ticker"]="UNIVERSE";MARKET["split"]=STOCK_DF.groupby("date")["split"].first().reindex(CAL)
MARKET=MARKET.reset_index()
FEATURE_DOCS.update(past_market_abs="Trailing H-day absolute mean stock log return, bps",past_dispersion="Trailing H-day stock log return cross-sectional standard deviation, bps")
SPECS={
 "excess_return":dict(label="Excess-return relationship",lhs=f"E[excess_log_return_i(t+{EXECUTION_LAG}:t+{EXECUTION_LAG+HORIZON}) | X_t]",unit="bps",level="stock-date",target=1e4*fwd.sub(fwd.mean(axis=1),axis=0),base=["mom_5","mom_20","vol_20","beta_60","volume_z_20"],anchor="mom_5",definition="1e4 times forward stock log return minus same-day universe mean"),
 "realized_risk":dict(label="Risk dynamics",lhs=f"E[RMS_daily_return_i(t+{EXECUTION_LAG}:t+{EXECUTION_LAG+HORIZON}) | X_t]",unit="bps/day",level="stock-date",target=risk,base=["past_rms_5","past_rms_20","vol_20","volume_z_20"],anchor="past_rms_20",definition="1e4 * sqrt(mean of squared daily log returns inside forward holding window)"),
 "volume_activity":dict(label="Trading-activity relationship",lhs=f"E[mean_volume_z_i(t+{EXECUTION_LAG+1}:t+{EXECUTION_LAG+HORIZON}) | X_t]",unit="z-score",level="stock-date",target=activity,base=["volume_z_20","ts_mean(volume_z_20, 5)","vol_20","mom_5"],anchor="volume_z_20",definition="Mean of future volume_z_20 values; not raw volume growth; upstream z-score must be trailing only"),
 "market_shock":dict(label="Market-shock relationship",lhs=f"E[abs(mean_i(forward_log_return_i)) | X_t; H={HORIZON}]",unit="bps",level="date",target=(1e4*fwd.mean(axis=1).abs()).where(market_valid),base=["past_market_abs","past_rms_5","past_rms_20","mom_20"],anchor="past_market_abs",definition="Absolute equal-weight mean forward stock log return in the sampled universe, times 1e4; not an index return"),
 "dispersion":dict(label="Cross-sectional dispersion relationship",lhs=f"E[std_i(forward_log_return_i) | X_t; H={HORIZON}]",unit="bps",level="date",target=(1e4*fwd.std(axis=1,ddof=1)).where(market_valid),base=["past_dispersion","past_rms_5","past_rms_20","mom_20"],anchor="past_dispersion",definition="Cross-sectional sample standard deviation of forward stock log returns, times 1e4")}
assert set(LAW_FAMILIES)<=set(SPECS) and len(LAW_FAMILIES)==len(set(LAW_FAMILIES))
# Shared backbone; each law family uses separate adapters and caches.
SHARED_HF={};STATES=[];SKIPPED=[]


In [ ]:
MODULES = {'engine': 'DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"\nclass Backend:\n    """All array operations used by the scorer (PyTorch, float32 on the GPU)."""\n    name = "torch"\n    INF = float("inf")\n\n    def arr(self, a):            return torch.as_tensor(np.asarray(a, dtype="float32"), device=DEVICE)\n    def nan(self, shape):        return torch.full(shape, float("nan"), device=DEVICE)\n    def zeros(self, shape):      return torch.zeros(shape, device=DEVICE)\n    def isnan(self, x):          return torch.isnan(x)\n    def where(self, c, a, b):    return torch.where(c, a, b)\n    def f(self, x):              return x.to(torch.float32)\n    def f64(self, x):            return x.to(torch.float64)\n    def sort_rows(self, x):      return torch.sort(x, dim=1).values\n    def ss_rows(self, s, x, right): return torch.searchsorted(s.contiguous(), x.contiguous(), right=right).to(torch.float32)\n    def sum(self, x, dim, keep=False): return x.sum(dim=dim, keepdim=keep)\n    def cumsum0(self, x):        return torch.cumsum(x, dim=0)\n    def cat0(self, a, b):        return torch.cat([a, b], dim=0)\n    def windows(self, x, n):     # (T, N) -> (T, N, n), window ends at t, NaN before the start\n        return self.cat0(self.nan((n - 1, x.shape[1])), x).unfold(0, n, 1)\n    def amax_last(self, x):      return x.max(dim=-1).values\n    def amin_last(self, x):      return x.min(dim=-1).values\n    def matmul(self, a, b):      return a @ b\n    def abs(self, x):            return torch.abs(x)\n    def sign(self, x):           return torch.sign(x)\n    def log1p(self, x):          return torch.log1p(x)\n    def sqrt(self, x):           return torch.sqrt(x)\n    def exp(self, x):            return torch.exp(x)\n    def maximum(self, a, b):     return torch.maximum(a, b)\n    def minimum(self, a, b):     return torch.minimum(a, b)\n    def clamp_min(self, x, v):   return torch.clamp(x, min=v)\n    def np(self, x):             return x.detach().float().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)\n    def is_arr(self, x):         return torch.is_tensor(x)\n\n\n\nclass NumpyBackend:\n    name="numpy"; INF=float("inf")\n    arr=staticmethod(lambda a:np.asarray(a,dtype=np.float32))\n    nan=staticmethod(lambda shape:np.full(shape,np.nan,dtype=np.float32))\n    zeros=staticmethod(lambda shape:np.zeros(shape,dtype=np.float32))\n    isnan=staticmethod(np.isnan); where=staticmethod(np.where)\n    f=staticmethod(lambda a:np.asarray(a,dtype=np.float32));f64=staticmethod(lambda a:np.asarray(a,dtype=np.float64))\n    sort_rows=staticmethod(lambda a:np.sort(a,axis=1))\n    ss_rows=staticmethod(lambda s,x,right:np.array([np.searchsorted(a,b,side="right" if right else "left") for a,b in zip(s,x)],dtype=np.float32))\n    sum=staticmethod(lambda a,dim,keep=False:a.sum(axis=dim,keepdims=keep))\n    cumsum0=staticmethod(lambda a:np.cumsum(a,axis=0));cat0=staticmethod(lambda a,b:np.concatenate([a,b],axis=0))\n    windows=staticmethod(lambda a,n:np.lib.stride_tricks.sliding_window_view(np.concatenate([np.full((n-1,a.shape[1]),np.nan),a]),n,axis=0))\n    amax_last=staticmethod(lambda a:a.max(axis=-1));amin_last=staticmethod(lambda a:a.min(axis=-1))\n    matmul=staticmethod(np.matmul);abs=staticmethod(np.abs);sign=staticmethod(np.sign)\n    log1p=staticmethod(np.log1p);sqrt=staticmethod(np.sqrt);exp=staticmethod(np.exp)\n    maximum=staticmethod(np.maximum);minimum=staticmethod(np.minimum)\n    clamp_min=staticmethod(lambda a,v:np.maximum(a,v));np=staticmethod(np.asarray)\n    is_arr=staticmethod(lambda a:isinstance(a,np.ndarray))\nB = Backend() if DEVICE == "cuda" else NumpyBackend()\nTICKERS = np.sort(df["ticker"].unique())\nT_, N_ = len(CAL), len(TICKERS)\nt_idx = CAL.get_indexer(df["date"])\nn_idx = pd.Index(TICKERS).get_indexer(df["ticker"])\n\n\ndef to_matrix(col):\n    m = np.full((T_, N_), np.nan, dtype="float32")\n    m[t_idx, n_idx] = df[col].to_numpy(dtype="float32")\n    return m\n\n\nFEAT = {f: B.arr(to_matrix(f)) for f in ALL_FEATURES}\nY5 = B.arr(to_matrix(TARGET))                                     # 5-day forward excess log return (IC target)\nR1 = B.arr(np.expm1(to_matrix("ret_1d")))                         # simple return of day t (close t-1 -> t)\n\n# sector one-hot (N x G); stocks without a sector have an all-zero row\nsec_codes = [KEPLER_SECTORS.index(TICKER_SECTOR[t]) if TICKER_SECTOR.get(t) in KEPLER_SECTORS else -1 for t in TICKERS]\nS_np = np.zeros((N_, len(KEPLER_SECTORS)), dtype="float32")\nfor i, c in enumerate(sec_codes):\n    if c >= 0:\n        S_np[i, c] = 1.0\nS_ONEHOT = B.arr(S_np)\nHAS_SECTOR = B.arr(S_np.sum(1))[None, :] > 0\n\nDATE_SPLIT = df.groupby("date")["split"].first().reindex(CAL).fillna("none").to_numpy()\nPERIOD_IDX = {p: np.flatnonzero(DATE_SPLIT == p) for p in ["train", "val", "test"]}\nfor p, ix in PERIOD_IDX.items():\n    print(f"{p:>5}: {len(ix)} days, {CAL[ix[0]].date()} -> {CAL[ix[-1]].date()}")\nprint(f"panel {T_} days x {N_} stocks x {len(FEAT)} features on {DEVICE}")\nCS_FUNCS = {"rank": 1, "zscore": 1, "demean": 1, "sector_neutral": 1}\nTS_FUNCS = {"delay": 2, "delta": 2, "ts_mean": 2, "ts_std": 2, "ts_sum": 2, "ts_max": 2, "ts_min": 2,\n            "ts_rank": 2, "ema": 2, "count": 2, "ts_corr": 3}              # the window is always the last argument\nEW_FUNCS = {"abs": 1, "sign": 1, "slog": 1, "sqrt_abs": 1, "max2": 2, "min2": 2, "gt": 2, "lt": 2, "where": 3}\nALL_FUNCS = {**CS_FUNCS, **TS_FUNCS, **EW_FUNCS}\nMAX_WINDOW, MAX_NODES = 60, 60\nNAN = float("nan")\n\nDSL_DOC = f"""Each factor is ONE Python-style expression built from:\n- feature names (listed above) and numbers; operators + - * / and unary -\n- cross-sectional (across all stocks on the same day): rank(x) percentile 0..1 | zscore(x) | demean(x) |\n  sector_neutral(x) = x minus its sector\'s mean that day\n- time-series (per stock, past and today only; n is an integer 1..{MAX_WINDOW}, always the LAST argument):\n  delay(x,n) | delta(x,n) = x - delay(x,n) | ts_mean(x,n) | ts_std(x,n) | ts_sum(x,n) | ts_max(x,n) | ts_min(x,n) |\n  ts_rank(x,n) = percentile of today\'s x within the last n days | ema(x,n) | ts_corr(x,y,n) | count(cond,n) = days with cond > 0\n- element-wise: abs | sign | slog(x) = sign*log(1+|x|) | sqrt_abs | max2(x,y) | min2(x,y) | gt(x,y), lt(x,y) give 1 or 0 |\n  where(cond,a,b) = a where cond > 0, else b\nRules: no other names, no keyword arguments, at most {MAX_NODES} AST nodes, at least one feature.\nA HIGH factor value means BUY (the portfolio holds the top 10% of stocks by factor value)."""\n\n\nclass FormulaError(ValueError):\n    pass\n\n\nclass _NoDoubleNeg(ast.NodeTransformer):\n    def visit_UnaryOp(self, node):\n        self.generic_visit(node)\n        if isinstance(node.op, ast.USub) and isinstance(node.operand, ast.UnaryOp) and isinstance(node.operand.op, ast.USub):\n            return node.operand.operand\n        return node\n\n\ndef parse_formula(formula, features):\n    if not isinstance(formula, str) or not formula.strip():\n        raise FormulaError("empty formula")\n    try:\n        tree = ast.parse(formula.strip(), mode="eval")\n    except SyntaxError as e:\n        raise FormulaError(f"syntax error: {e.msg}")\n    n_nodes, uses_feature = 0, False\n    for node in ast.walk(tree):\n        n_nodes += 1\n        if isinstance(node, (ast.Expression, ast.Load, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.USub, ast.UAdd,\n                             ast.BinOp, ast.UnaryOp)):\n            continue\n        if isinstance(node, ast.Constant):\n            if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):\n                raise FormulaError(f"constant {node.value!r} is not a number")\n            continue\n        if isinstance(node, ast.Name):\n            if node.id in LEAK_COLS:\n                raise FormulaError(f"\'{node.id}\' is the target / future information")\n            if node.id in features:\n                uses_feature = True\n            elif node.id not in ALL_FUNCS:\n                raise FormulaError(f"unknown name \'{node.id}\'")\n            continue\n        if isinstance(node, ast.Call):\n            fn = node.func.id if isinstance(node.func, ast.Name) else ast.unparse(node.func)\n            if fn not in ALL_FUNCS:\n                raise FormulaError(f"function \'{fn}\' is not allowed")\n            if node.keywords:\n                raise FormulaError(f"{fn}(): keyword arguments are not allowed")\n            if len(node.args) != ALL_FUNCS[fn]:\n                raise FormulaError(f"{fn}() takes {ALL_FUNCS[fn]} argument(s)")\n            if fn in TS_FUNCS:\n                w = node.args[-1]\n                if not (isinstance(w, ast.Constant) and isinstance(w.value, int) and not isinstance(w.value, bool)\n                        and 1 <= w.value <= MAX_WINDOW):\n                    raise FormulaError(f"{fn}(): window (last argument) must be an integer 1..{MAX_WINDOW}")\n            continue\n        raise FormulaError(f"\'{type(node).__name__}\' is not allowed")\n    if n_nodes > MAX_NODES:\n        raise FormulaError(f"too complex ({n_nodes} nodes > {MAX_NODES})")\n    if not uses_feature:\n        raise FormulaError("formula uses no feature")\n    return _NoDoubleNeg().visit(tree)\n\n\ndef canonical(formula, features=None):\n    return ast.unparse(parse_formula(formula, features or ALL_FEATURES))\ndef _ser(v):\n    return v if B.is_arr(v) else B.arr(np.full((T_, N_), float(v), dtype="float32"))\n\n\ndef _clean(x):\n    return B.where(B.abs(x) == B.INF, B.nan(x.shape), x)\n\n\ndef row_mean(x):\n    v = ~B.isnan(x)\n    s = B.sum(B.where(v, x, B.zeros(x.shape)), 1, keep=True)\n    c = B.sum(B.f(v), 1, keep=True)\n    return s / B.where(c > 0, c, B.nan(c.shape)), c\n\n\ndef cs_rank(x):\n    """Percentile rank per row (average rank for ties), NaN stays NaN."""\n    nanm = B.isnan(x)\n    xf = B.where(nanm, B.zeros(x.shape) + B.INF, x)\n    s = B.sort_rows(xf)\n    lo, hi = B.ss_rows(s, xf, False), B.ss_rows(s, xf, True)\n    cnt = B.sum(B.f(~nanm), 1, keep=True)\n    r = ((lo + hi - 1.0) / 2.0 + 1.0) / B.where(cnt > 0, cnt, B.nan(cnt.shape))\n    return B.where(nanm, B.nan(x.shape), r)\n\n\ndef _pad_front(C, n):\n    T1 = C.shape[0]\n    if n >= T1:\n        return B.f64(B.zeros(C.shape))\n    return B.cat0(B.f64(B.zeros((n, C.shape[1]))), C[:T1 - n])\n\n\ndef _roll(xs, n):\n    """Rolling sums over the last n rows of several NaN-free float64 arrays."""\n    out = []\n    for x in xs:\n        C = B.cumsum0(B.cat0(B.f64(B.zeros((1, x.shape[1]))), x))\n        out.append((C - _pad_front(C, n))[1:])\n    return out\n\n\ndef ts_basic(x, n, kind):\n    minp = max(2, n // 2)\n    v = ~B.isnan(x)\n    xf = B.f64(B.where(v, x, B.zeros(x.shape)))\n    S, SS, Cn = _roll([xf, xf * xf, B.f64(B.f(v))], n)\n    ok = Cn >= minp\n    if kind == "sum":\n        out = S\n    elif kind == "mean":\n        out = S / B.where(Cn > 0, Cn, Cn + 1)\n    else:\n        var = (SS - S * S / B.where(Cn > 0, Cn, Cn + 1)) / B.where(Cn > 1, Cn - 1, Cn + 1)\n        out = B.sqrt(B.clamp_min(var, 0.0))\n    return B.where(ok, B.f(out), B.nan(x.shape))\n\n\ndef ts_extreme(x, n, kind):\n    minp = max(2, n // 2)\n    W = B.windows(x, n)\n    valid = ~B.isnan(W)\n    cnt = B.sum(B.f(valid), -1)\n    fill = (B.zeros(W.shape) - B.INF) if kind == "max" else (B.zeros(W.shape) + B.INF)\n    Wf = B.where(valid, W, fill)\n    m = B.amax_last(Wf) if kind == "max" else B.amin_last(Wf)\n    return B.where(cnt >= minp, m, B.nan(x.shape))\n\n\ndef ts_rank(x, n):\n    minp = max(2, n // 2)\n    W = B.windows(x, n)\n    valid = ~B.isnan(W)\n    cnt = B.sum(B.f(valid), -1)\n    le = B.sum(B.f((W <= x[:, :, None]) & valid), -1)\n    out = le / B.where(cnt > 0, cnt, cnt + 1)\n    return B.where((cnt >= minp) & ~B.isnan(x), out, B.nan(x.shape))\n\n\ndef ts_corr(x, y, n):\n    minp = max(3, n // 2)\n    v = ~B.isnan(x) & ~B.isnan(y)\n    xf = B.f64(B.where(v, x, B.zeros(x.shape)))\n    yf = B.f64(B.where(v, y, B.zeros(y.shape)))\n    Sx, Sy, Sxx, Syy, Sxy, Cn = _roll([xf, yf, xf * xf, yf * yf, xf * yf, B.f64(B.f(v))], n)\n    Cs = B.where(Cn > 0, Cn, Cn + 1)\n    cov, vx, vy = Sxy - Sx * Sy / Cs, Sxx - Sx * Sx / Cs, Syy - Sy * Sy / Cs\n    ok = (Cn >= minp) & (vx > 1e-12) & (vy > 1e-12)\n    den = B.sqrt(B.clamp_min(vx * vy, 1e-30))\n    return B.where(ok, B.f(cov / den), B.nan(x.shape))\n\n\ndef ema(x, n):\n    a = 2.0 / (n + 1.0)\n    out = B.nan(x.shape)\n    prev = B.nan((x.shape[1],))\n    for t in range(x.shape[0]):\n        xt = x[t]\n        new = B.where(B.isnan(xt), prev, B.where(B.isnan(prev), xt, a * xt + (1 - a) * prev))\n        out[t] = new\n        prev = new\n    return out\n\n\ndef shift(x, n):\n    return B.cat0(B.nan((n, x.shape[1])), x[:-n]) if n < x.shape[0] else B.nan(x.shape)\n\n\ndef sector_neutral(x):\n    v = ~B.isnan(x)\n    xf = B.where(v, x, B.zeros(x.shape))\n    gs, gc = B.matmul(xf, S_ONEHOT), B.matmul(B.f(v), S_ONEHOT)\n    gm = gs / B.where(gc > 0, gc, gc + 1)\n    m = B.matmul(B.where(gc > 0, gm, B.zeros(gm.shape)), S_ONEHOT.T)\n    return B.where(v & HAS_SECTOR, x - m, B.nan(x.shape))\n\n\ndef _both_nan(a, b):\n    return B.isnan(a) | B.isnan(b)\n\n\nclass GPUEngine:\n    def __init__(self, features, store, tag="main"):\n        self.features, self.store, self.tag = set(features), store, tag\n\n    def evaluate(self, formula):\n        tree = parse_formula(formula, self.features)\n        out = self._eval(tree.body)\n        if not B.is_arr(out):\n            raise FormulaError("formula evaluates to a constant")\n        return _clean(out)\n\n    def _eval(self, node):\n        if isinstance(node, ast.Constant):\n            return float(node.value)\n        if isinstance(node, ast.Name):\n            return self.store[node.id]\n        if isinstance(node, ast.UnaryOp):\n            v = self._eval(node.operand)\n            return -v if isinstance(node.op, ast.USub) else v\n        if isinstance(node, ast.BinOp):\n            a, b = self._eval(node.left), self._eval(node.right)\n            if isinstance(node.op, ast.Add):\n                return a + b\n            if isinstance(node.op, ast.Sub):\n                return a - b\n            if isinstance(node.op, ast.Mult):\n                return a * b\n            if B.is_arr(b):\n                return a / B.where(B.abs(b) > 1e-12, b, B.nan(b.shape))\n            return a / b if abs(b) > 1e-12 else NAN\n        fn = node.func.id\n        if fn in TS_FUNCS:\n            n = node.args[-1].value\n            x = _ser(self._eval(node.args[0]))\n            if fn == "delay":\n                return shift(x, n)\n            if fn == "delta":\n                return x - shift(x, n)\n            if fn in ("ts_mean", "ts_sum", "ts_std"):\n                return ts_basic(x, n, fn[3:])\n            if fn in ("ts_max", "ts_min"):\n                return ts_extreme(x, n, fn[3:])\n            if fn == "ts_rank":\n                return ts_rank(x, n)\n            if fn == "ema":\n                return ema(x, n)\n            if fn == "count":\n                c = B.where(B.isnan(x), B.nan(x.shape), B.f(x > 0))\n                return ts_basic(c, n, "sum")\n            if fn == "ts_corr":\n                return ts_corr(x, _ser(self._eval(node.args[1])), n)\n        args = [self._eval(a) for a in node.args]\n        if fn in CS_FUNCS:\n            x = _ser(args[0])\n            if fn == "rank":\n                return cs_rank(x)\n            if fn == "sector_neutral":\n                return sector_neutral(x)\n            mu, c = row_mean(x)\n            if fn == "demean":\n                return x - mu\n            d = B.where(B.isnan(x), B.zeros(x.shape), (x - mu) ** 2)\n            sd = B.sqrt(B.sum(d, 1, keep=True) / B.where(c > 1, c - 1, c + 1))\n            return (x - mu) / B.where(sd > 1e-12, sd, B.nan(sd.shape))\n        if fn in ("abs", "sign", "slog", "sqrt_abs"):\n            x = args[0]\n            if not B.is_arr(x):\n                x = _ser(x)\n            if fn == "abs":\n                return B.abs(x)\n            if fn == "sign":\n                return B.sign(x)\n            if fn == "slog":\n                return B.sign(x) * B.log1p(B.abs(x))\n            return B.sign(x) * B.sqrt(B.abs(x))\n        if fn == "where":\n            c, a, b = _ser(args[0]), _ser(args[1]), _ser(args[2])\n            return B.where(B.isnan(c), B.nan(c.shape), B.where(c > 0, a, b))\n        a, b = _ser(args[0]), _ser(args[1])\n        if fn == "max2":\n            return B.maximum(a, b)\n        if fn == "min2":\n            return B.minimum(a, b)\n        r = B.f(a > b) if fn == "gt" else B.f(a < b)\n        return B.where(_both_nan(a, b), B.nan(a.shape), r)\n\n\nENG = GPUEngine(ALL_FEATURES, FEAT)\nENG_PRICE = GPUEngine(PRICE_FEATURES, FEAT, tag="price")\n\n# Term DSL intentionally excludes rank/zscore/demean/sector_neutral, so the model estimates a scale-bearing relationship.\nTERM_FUNCTIONS = {k:v for k,v in ALL_FUNCS.items() if k not in CS_FUNCS}\nTERM_CACHE={}\ndef term_features(term):\n    return {a.id for a in ast.walk(ast.parse(term,mode="eval")) if isinstance(a,ast.Name) and a.id in FEATURES}\ndef term_uses_news(term): return any(f.startswith("world_") for f in term_features(term))\ndef canonical_term(text):\n    if not isinstance(text,str) or len(text)>500: raise FormulaError("Term must be a short expression")\n    tree=parse_formula(text,FEATURES)\n    for n in ast.walk(tree):\n        if isinstance(n,ast.Call) and n.func.id not in TERM_FUNCTIONS:\n            raise FormulaError("rank/normalization score functions excluded: use raw or transformed input variables")\n        if isinstance(n,ast.Constant) and (not np.isfinite(n.value) or abs(n.value)>1e4):\n            raise FormulaError("invalid or oversized constant")\n    return ast.unparse(tree)\n\ndef term_values(term):\n    term=canonical_term(term)\n    if term not in TERM_CACHE:\n        a=B.np(ENG.evaluate(term))[t_idx,n_idx].astype(np.float32)\n        a[~np.isfinite(a)]=np.nan\n        TERM_CACHE[term]=a\n    return TERM_CACHE[term]\n\ndef canonical_structure(terms):\n    if not isinstance(terms,list) or not 1<=len(terms)<=MAX_TERMS: raise FormulaError("Need 1..MAX_TERMS terms")\n    out={canonical_term(t) for t in terms}\n    # Strong heredity for simple feature-by-feature interactions: include both main effects.\n    for t in list(out):\n        n=ast.parse(t,mode="eval").body\n        if isinstance(n,ast.BinOp) and isinstance(n.op,ast.Mult) and isinstance(n.left,ast.Name) and isinstance(n.right,ast.Name):\n            out.update([n.left.id,n.right.id])\n    if len(out)>MAX_TERMS: raise FormulaError("Too many terms after including interaction main effects")\n    return tuple(sorted(out))\n', 'splits': 'Y = df[TARGET].to_numpy(float)   # bps; no sign orientation, no ranking of y\nDAY = CAL.get_indexer(df.date)\nLABEL_OK = np.isfinite(Y)\nfor p in ["train","val","test"]:\n    raw=SPLIT_MASK[p] & LABEL_OK\n    counts=np.bincount(DAY[raw],minlength=len(CAL))\n    SPLIT_MASK[p] = raw & (counts[DAY]>=MIN_NAMES)\nIDX={p:np.flatnonzero(SPLIT_MASK[p]) for p in ["train","val","test"]}\nfor p,ix in IDX.items():\n    if len(np.unique(DAY[ix]))<60: raise ValueError(f"Insufficient {p} label dates")\nTRAIN_DAYS=np.unique(DAY[IDX["train"]]); GAP=HORIZON+EXECUTION_LAG\nedges=np.linspace(int(.4*len(TRAIN_DAYS)),len(TRAIN_DAYS),CV_FOLDS+1,dtype=int)\nFOLDS=[]\nfor k in range(CV_FOLDS):\n    vd=TRAIN_DAYS[edges[k]:edges[k+1]]\n    tr=IDX["train"][DAY[IDX["train"]]<vd[0]-GAP]\n    va=IDX["train"][np.isin(DAY[IDX["train"]],vd)]\n    if len(np.unique(DAY[tr]))>=CV_MIN_TRAIN_DAYS and len(vd)>=30: FOLDS.append((tr,va))\nif len(FOLDS)<2: raise ValueError("Need >=2 expanding CV folds inside TRAIN")\nprint("Rows / days:",{p:(len(ix),len(np.unique(DAY[ix]))) for p,ix in IDX.items()})\nprint("CV folds:",[(len(np.unique(DAY[a])),len(np.unique(DAY[b]))) for a,b in FOLDS])\nprint("All candidates use the SAME label rows; zero news and missing term values do not remove outcome dates.")\n\n# Audit reproducibility / cache identity. Do not silently reuse results under changed data or settings.\ndef hash_file(p):\n    h=hashlib.sha256()\n    with open(p,"rb") as f:\n        for b in iter(lambda:f.read(8*1024*1024),b""):h.update(b)\n    return h.hexdigest()\nconfig_names="SEED STUDY_START VAL_START TEST_START STUDY_END HORIZON EXECUTION_LAG NEWS_LAG MAX_TICKERS HOLDOUT_IS_FRESH MAX_TERMS RIDGE_LAMBDA CV_FOLDS CV_MIN_TRAIN_DAYS MIN_TERM_COVERAGE ROBUST_LAMBDA COMPLEXITY_PENALTY PRICE_BASELINE_FEATURES EQUIVALENCE_RHO HF_MODEL_ID USE_LLM BASE_ROUNDS POST_ROUNDS PROPOSALS_PER_ROUND MAX_NEW_TOKENS TEMPERATURE RUN_BASE_CONTROL RANDOM_CANDIDATES OBJECTIVES TRAIN_PROMPTS PROMPTS_PER_STEP TRAIN_EPOCHS TARGETS_PER_ANSWER TRAIN_MAX_TOKENS TRAIN_LR LORA_R LORA_ALPHA MIN_TRAIN_SCORE DPO_BETA GRPO_BETA GRPO_K REWARD_SCALE TOP_FINAL BOOTSTRAPS COEF_BOOTSTRAPS BOOT_BLOCK NW_LAGS MIN_TEST_DAYS MIN_CV_CONSISTENCY MAX_TEST_IMPUTED_SHARE".split()\nMANIFEST={"implementation":"multi_laws_v3_relative_sft", "family":FAMILY, "definition":SPEC["definition"], "families":LAW_FAMILIES, "demo":DEMO_MODE,"config":{k:globals()[k] for k in config_names},\n          "prices":"SYNTHETIC" if DEMO_MODE else hash_file(data_path),"news":hash_file(factors_path),\n          "sector_map":sorted(TICKER_SECTOR.items()),"tickers":TICKERS.tolist()}\nRUN_HASH=hashlib.sha256(json.dumps(MANIFEST,sort_keys=True,default=str).encode()).hexdigest()\nmp=Path(OUTPUT_DIR)/"manifest.json"\nif mp.exists() and json.loads(mp.read_text())["run_hash"]!=RUN_HASH:\n    raise RuntimeError("Data/settings changed: choose another RUN_NAME. The old historical test remains previously seen.")\nmp.write_text(json.dumps({"run_hash":RUN_HASH,"manifest":MANIFEST},indent=2,default=str))\n\nassert pd.Timestamp(NEWS_CAP_END) <= CAL[min(DAY[va].min()-GAP for tr,va in FOLDS)], "News caps extend beyond this family inner training prefix"\n', 'regression': 'def matrix_terms(terms, indices):\n    return np.column_stack([term_values(t)[indices] for t in terms]).astype(float) if terms else np.empty((len(indices),0))\n\ndef day_weights(indices):\n    days=DAY[indices]; counts=np.bincount(days,minlength=len(CAL)); nd=np.count_nonzero(counts)\n    return 1.0/counts[days]/nd\n\ndef fit_equation(terms, indices):\n    terms=tuple(terms); X=matrix_terms(terms,indices); y=Y[indices]; w=day_weights(indices)\n    if len(terms):\n        coverage=np.isfinite(X).mean(axis=0)\n        if np.any(coverage<MIN_TERM_COVERAGE): raise FormulaError("Insufficient term coverage in this TRAIN fold")\n        med=np.nanmedian(X,axis=0); X=np.where(np.isfinite(X),X,med)\n        if np.max(np.abs(X))>1e10: raise FormulaError("Numerically unstable term magnitude on TRAIN")\n        mu=w@X; scale=np.sqrt(w@((X-mu)**2));scale=np.where(scale>1e-10,scale,1.)\n        Z=(X-mu)/scale; ym=float(w@y)\n        G=Z.T@(Z*w[:,None]); rhs=Z.T@(w*(y-ym))\n        b=np.linalg.solve(G+RIDGE_LAMBDA*np.eye(len(terms)),rhs)\n        coef=b/scale; intercept=ym-float(mu@coef)\n    else:\n        med=mu=scale=coef=np.array([]);intercept=float(w@y)\n    return dict(terms=list(terms),coef=coef.tolist(),intercept=intercept,medians=med.tolist(),\n                means=mu.tolist(),scales=scale.tolist(),fit_rows=len(indices),fit_days=len(np.unique(DAY[indices])))\n\ndef predict_equation(model,indices):\n    X=matrix_terms(model["terms"],indices)\n    if X.shape[1]: X=np.where(np.isfinite(X),X,np.asarray(model["medians"]))\n    pred=model["intercept"]+X@np.asarray(model["coef"])\n    if not np.isfinite(pred).all(): raise FormulaError("Nonfinite predictions")\n    return pred\n\ndef daily_mse(pred,indices):\n    days=DAY[indices]; error=(Y[indices]-pred)**2\n    if not np.isfinite(error).all(): raise FormulaError("Nonfinite squared errors")\n    num=np.bincount(days,weights=error,minlength=len(CAL));den=np.bincount(days,minlength=len(CAL));keep=den>0\n    return pd.Series(num[keep]/den[keep],index=CAL[keep])\n\ndef loss_of(model,indices): return daily_mse(predict_equation(model,indices),indices)\ndef ratio_gain(loss,reference):\n    if not loss.index.equals(reference.index):raise AssertionError("Paired comparison requires exactly the same dates")\n    den=float(reference.mean())\n    return 1-float(loss.mean())/den if den>0 else np.nan\n\ndef input_imputation_share(model,indices):\n    X=matrix_terms(model["terms"],indices)\n    return float((~np.isfinite(X)).any(axis=1).mean()) if X.shape[1] else 0.\n\ndef without_news(terms):return tuple(t for t in terms if not term_uses_news(t))\nBASE_TERMS=tuple(PRICE_BASELINE_FEATURES)\nBASE_FOLDS=[fit_equation(BASE_TERMS,tr) for tr,va in FOLDS]\nBASE_CV_LOSS=[loss_of(m,va) for m,(tr,va) in zip(BASE_FOLDS,FOLDS)]\nBASE_MODEL=fit_equation(BASE_TERMS,IDX["train"])\nCV_CACHE={};MODEL_CACHE={};RECORDS=[]\n\ndef score_structure(terms):\n    terms=canonical_structure(list(terms)); key=json.dumps(terms)\n    if key not in CV_CACHE:\n        gains=[];losses=[]; coefs=[]\n        for (tr,va),base_loss in zip(FOLDS,BASE_CV_LOSS):\n            model=fit_equation(terms,tr);loss=loss_of(model,va)\n            gains.append(ratio_gain(loss,base_loss));losses.append(float(loss.mean()));coefs.append(model["coef"])\n        g=np.asarray(gains);complexity=sum(len(list(ast.walk(ast.parse(t,mode="eval")))) for t in terms)\n        CV_CACHE[key]=dict(terms=list(terms),cv_gain=float(g.mean()),cv_gain_sd=float(g.std(ddof=1)),\n            cv_consistency=float((g>0).mean()),cv_mse=float(np.mean(losses)),fold_gains=g.tolist(),fold_coefs=coefs,\n            robust=float(g.mean()-ROBUST_LAMBDA*g.std(ddof=1)-COMPLEXITY_PENALTY*len(terms)),\n            complexity=complexity,uses_news=any(term_uses_news(t) for t in terms))\n    return dict(CV_CACHE[key])\n\ndef fitted(terms):\n    key=json.dumps(tuple(terms))\n    if key not in MODEL_CACHE:MODEL_CACHE[key]=fit_equation(terms,IDX["train"])\n    return MODEL_CACHE[key]\n\ndef record_proposal(prop,source,seen):\n    rec=dict(source=source,name=str(prop.get("name","")),hypothesis=str(prop.get("hypothesis","")),status="error")\n    try:\n        terms=canonical_structure(prop.get("terms"));key=json.dumps(terms);rec.update(terms=list(terms),key=key)\n        if key in seen:rec.update(status="duplicate");return rec\n        rec.update(score_structure(terms),status="ok")\n    except (ValueError,TypeError,KeyError,ArithmeticError,np.linalg.LinAlgError) as exc:\n        rec["error"]=str(exc)\n    return rec\n\ndef persist_records():\n    pd.DataFrame(RECORDS).to_json(Path(OUTPUT_DIR)/"search_records.json",orient="records",indent=2,force_ascii=False)\n', 'search': 'anchor=SPEC["anchor"]\nnews_candidates=[x for x in ["world_surprise","world_mood_surprise","world_fear","world_shock","world_war","world_count"] if x in FEATURES]\nSEED_STRUCTURES=[list(BASE_TERMS),[anchor],[anchor,f"slog({anchor})"]]\nfor nf in news_candidates:\n    SEED_STRUCTURES += [[anchor,nf,f"{anchor} * {nf}"],list(BASE_TERMS)+[nf]]\n_seen=set()\nfor terms in SEED_STRUCTURES:\n    rec=record_proposal({"name":"seed_structure","terms":terms,"hypothesis":"Predefined seed; not an LLM discovery"},"seed",_seen)\n    RECORDS.append(rec)\n    if rec.get("key"):_seen.add(rec["key"])\n# Structure-level random baseline, with same coefficient fitter and CV objective.\nrr=random.Random(SEED)\nprice=[f for f in PRICE_BASELINE_FEATURES if f in FEATURES] or [anchor]\nnews=[f for f in FEATURES if f.startswith("world_")]\nfor i in range(RANDOM_CANDIDATES):\n    a=rr.choice(news if news and i%2==0 else price);b=rr.choice(price)\n    terms=[a,b,f"{a} * {b}"] if a!=b and i%3 else [a, f"slog({b})"]\n    rec=record_proposal({"name":"random_structure","terms":terms,"hypothesis":"Random baseline"},"random",_seen)\n    RECORDS.append(rec)\n    if rec.get("key"):_seen.add(rec["key"])\npersist_records()\nvalid=[r for r in RECORDS if r["status"]=="ok"]\nif not valid:raise RuntimeError("No valid seed structure: inspect feature coverage and dates")\ndisplay(pd.DataFrame(valid).sort_values("robust",ascending=False)[["source","terms","cv_gain","cv_consistency","robust"]].head(8))\nprint("Early equations scored; all scores above come from TRAIN-CV only.")\n', 'llm': 'SYSTEM="You propose small, interpretable regression equation structures. Return JSON only. Never invent fitted coefficients or claim causality."\nFEATURE_TEXT="\\n".join(f"{f}: {FEATURE_DOCS.get(f,f)}" for f in FEATURES)\nTERM_DSL="Raw feature names and + - * /; functions: abs, slog, sqrt_abs (signed square root), delay(x,n), delta(x,n), ts_mean(x,n), ts_std(x,n), ts_corr(x,y,n), where(cond,a,b), gt(x,y), lt(x,y). Integer windows 1..60. No rank/zscore. At most six terms; coefficients/intercept are fitted separately. Include both main effects with a feature*feature interaction."\n\ndef build_prompt(context,n=PROPOSALS_PER_ROUND):\n    ok=[r for r in context if r.get("status")=="ok"]\n    best=sorted(ok,key=lambda r:-r["robust"])[:6]\n    feedback="\\n".join(json.dumps({k:r[k] for k in ["terms","cv_gain","cv_consistency","robust"]}) for r in best)\n    errors="\\n".join(str(r.get("error","")) for r in context if r.get("status")=="error")[-350:]\n    return f"""Target relation: {SPEC["lhs"]}. Unit: {SPEC["unit"]}. Sampling level: {SPEC["level"]}. Definition: {SPEC["definition"]}.\nFor each proposal we fit this named conditional mean = intercept + sum(coef_j * term_j). Coefficients are estimated using TRAIN only.\nObjective: reduce date-equal MSE versus a fixed price-only regression in purged forward CV; favor gains that persist across folds. These are TRAIN-CV results, not test.\nFor the excess_return family, shared market news needs stock-specific interactions. Other families may have a market-wide response. Do not propose algebraic identities or future inputs.\nFeatures (news is already lagged):\n{FEATURE_TEXT}\nDSL: {TERM_DSL}\nBest training-only structures:\n{feedback}\nRecent rejection notes: {errors}\nReturn {n} distinct compact equation structures, as a JSON list:\n[{{"name":"brief_name","terms":["world_surprise","mom_5","world_surprise * mom_5"],"hypothesis":"a falsifiable statistical relationship"}}]\nDo not return coefficients, future columns, code fences, or prose outside JSON."""\n\ndef parse_structures(text):\n    text=re.sub(r"<think>.*?</think>","",text,flags=re.S)\n    decoder=json.JSONDecoder()\n    for m in re.finditer(r"\\[",text):\n        try:\n            obj,_=decoder.raw_decode(text[m.start():])\n            if isinstance(obj,list):\n                return [x for x in obj if isinstance(x,dict) and isinstance(x.get("terms"),list)]\n        except json.JSONDecodeError:pass\n    # Recover individual complete objects if a long list was truncated.\n    found=[]\n    for m in re.finditer(r"\\{",text):\n        try:\n            obj,_=decoder.raw_decode(text[m.start():])\n            if isinstance(obj,dict) and isinstance(obj.get("terms"),list):found.append(obj)\n        except json.JSONDecodeError:pass\n    return found\n\nHF=SHARED_HF\nif USE_LLM and not HF:\n    if torch is None or not torch.cuda.is_available():raise RuntimeError("For LLM training choose a Colab GPU, or set USE_LLM=False for template-only equation search")\n    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig\n    TOK=AutoTokenizer.from_pretrained(HF_MODEL_ID)\n    if TOK.pad_token_id is None:TOK.pad_token=TOK.eos_token\n    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16)\n    HF["model"]=AutoModelForCausalLM.from_pretrained(HF_MODEL_ID,device_map={"":0},quantization_config=quant).eval()\n    HF["tok"]=TOK\n    print("Model loaded:",HF_MODEL_ID)\n\n# The shared backbone already exists after the first family; always bind its tokenizer.\nif USE_LLM: TOK=HF["tok"]\n\n@contextlib.contextmanager\ndef adapter_context(name=None):\n    model=HF["model"]\n    if name is None and hasattr(model,"disable_adapter"):\n        with model.disable_adapter():yield\n    else:\n        if name is not None:model.set_adapter(name)\n        yield\n\ndef generate_structures(prompt,adapter=None,n_sequences=1,return_tokens=False):\n    tok,m=HF["tok"],HF["model"]\n    text=tok.apply_chat_template([{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],tokenize=False,add_generation_prompt=True)\n    enc=tok(text,return_tensors="pt",add_special_tokens=False).to(m.device)\n    m.eval();m.config.use_cache=True\n    with torch.no_grad(),adapter_context(adapter):\n        out=m.generate(**enc,do_sample=True,temperature=TEMPERATURE,top_p=.95,top_k=64,\n                       max_new_tokens=MAX_NEW_TOKENS,num_return_sequences=n_sequences,pad_token_id=tok.pad_token_id)\n    gens=out[:,enc.input_ids.shape[1]:]\n    texts=tok.batch_decode(gens,skip_special_tokens=True)\n    return (texts,enc.input_ids[0],gens) if return_tokens else texts\n\ndef run_search(source,start,rounds,adapter=None):\n    cache=Path(OUTPUT_DIR)/f"search_{source}.json"\n    if RESUME and cache.exists():\n        saved=json.loads(cache.read_text())\n        if saved["run_hash"]!=RUN_HASH:raise RuntimeError("Search cache hash mismatch")\n        out=saved["records"]\n        for r in out:\n            if r["status"]=="ok":r.update(score_structure(r["terms"]))\n        RECORDS.extend(out);return out\n    context=copy.deepcopy(start);seen={r["key"] for r in context if "key" in r};new=[]\n    torch.manual_seed(SEED+100)\n    for rnd in range(rounds):\n        txt=generate_structures(build_prompt(context),adapter=adapter)[0]\n        props=parse_structures(txt)[:PROPOSALS_PER_ROUND]\n        batch=[]\n        for p in props:\n            rec=record_proposal(p,source,seen);rec["round"]=rnd+1;batch.append(rec)\n            if rec.get("key"):seen.add(rec["key"])\n        context.extend(batch);new.extend(batch)\n        (Path(OUTPUT_DIR)/f"raw_{source}_{rnd+1}.txt").write_text(txt)\n        print(source,rnd+1,Counter(r["status"] for r in batch))\n    RECORDS.extend(new);persist_records()\n    cache.write_text(json.dumps({"run_hash":RUN_HASH,"records":new},indent=2))\n    return new\n\n# Random proposals are NOT used as LLM feedback or training targets, preserving baseline separation.\nINITIAL=[r for r in RECORDS if r["source"]=="seed"]\nif USE_LLM:run_search("base_initial",INITIAL,BASE_ROUNDS)\nPRETRAIN_CONTEXT=copy.deepcopy([r for r in RECORDS if r["source"] in ["seed","base_initial"]])\n', 'training': 'TRAINED=[];TRAIN_LOGS={};TRAINING_NOTES=[]\n\ndef choose_training_targets(context):\n    # Only TRAIN-CV records enter this function. No VAL/TEST scores or final selections.\n    pool=list({r["key"]:r for r in context if r.get("status")=="ok" and\n               np.isfinite(r.get("robust",np.nan)) and np.isfinite(r.get("cv_gain",np.nan))}.values())\n    ranked=sorted(pool,key=lambda r:(-r["robust"],len(r["terms"])))\n    good=[r for r in ranked if r["robust"]>MIN_TRAIN_SCORE]\n    def diverse(rows,limit):\n        picked=[]\n        for r in rows:\n            a=set(r["terms"])\n            if all(len(a & set(p["terms"]))/max(1,len(a | set(p["terms"])))<.8 for p in picked):picked.append(r)\n            if len(picked)>=limit:break\n        return picked\n    selected=diverse(good,6)\n    mode="positive_cv_expert_iteration"\n    if len(selected)<2:\n        # Relative quality is sufficient for learning. It is NOT evidence of a valid law.\n        top=ranked[:min(len(ranked),max(2,int(np.ceil(.3*len(ranked)))))]\n        selected=diverse(top,6)\n        if len(selected)<min(2,len(top)):\n            used={r["key"] for r in selected}\n            selected += [r for r in top if r["key"] not in used][:2-len(selected)]\n        mode="relative_cv_weak_supervision"\n    used={r["key"] for r in selected}\n    negatives=sorted([r for r in pool if r["key"] not in used],key=lambda r:r["robust"])\n    # DPO needs an actual preference gap; SFT never requires negatives.\n    if selected:negatives=[r for r in negatives if r["robust"]<min(p["robust"] for p in selected)-1e-12]\n    return pool,good,selected,negatives[:max(2,len(negatives)//3)],mode\n\nPOOL,GOOD,POS,NEG,TRAINING_MODE=choose_training_targets(PRETRAIN_CONTEXT)\nDIAGNOSTICS=dict(llm_enabled=bool(USE_LLM),valid_candidates=len(POOL),absolute_positive_candidates=len(GOOD),\n                 selected_training_targets=len(POS),preference_negatives=len(NEG),mode=TRAINING_MODE,\n                 target_scores=[dict(terms=r["terms"],cv_gain=r["cv_gain"],robust=r["robust"]) for r in POS])\nPath(OUTPUT_DIR,"training_pool_diagnostics.json").write_text(json.dumps(DIAGNOSTICS,indent=2))\nprint("LoRA training pool:",json.dumps({k:v for k,v in DIAGNOSTICS.items() if k!="target_scores"}))\nREADY=USE_LLM and bool(POS)\nif not READY:\n    reason="USE_LLM=False (including synthetic demo mode)" if not USE_LLM else "no valid finite TRAIN-CV structures"\n    TRAINING_NOTES.append("LoRA skipped: "+reason)\n    print(TRAINING_NOTES[-1])\nelse:\n    TRAINING_NOTES.append("Supervision mode: "+TRAINING_MODE+". Training eligibility is separate from law validation.")\n    from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training\n    TARGET_POOLS={"ei_robust":POS,"ei_mse":sorted(POS,key=lambda r:-r["cv_gain"])}\n    CTX=[r for r in PRETRAIN_CONTEXT if r.get("key") not in {p["key"] for p in POS}]\n    def answer(items):return json.dumps([{k:r[k] for k in ["name","terms","hypothesis"]} for r in items])\n    def encode(prompt,response):\n        msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}]\n        prefix=TOK.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)\n        full=TOK.apply_chat_template(msgs+[{"role":"assistant","content":response}],tokenize=False)\n        if not full.startswith(prefix):raise RuntimeError("Chat template prefix mismatch")\n        pi=TOK(prefix,add_special_tokens=False)["input_ids"]\n        ai=TOK(full[len(prefix):],add_special_tokens=False)["input_ids"]\n        return dict(ids=pi+ai,n_ans=len(ai))\n    EXAMPLES=[];rr=random.Random(SEED)\n    for i in range(TRAIN_PROMPTS):\n        prompt=build_prompt(rr.sample(CTX,min(3,len(CTX))),TARGETS_PER_ANSWER)\n        ex={"prompt":prompt}\n        for key,pool in [("robust",POS),("mse",TARGET_POOLS["ei_mse"]),("neg",NEG)]:\n            ex[key]=encode(prompt,answer(rr.sample(pool,min(TARGETS_PER_ANSWER,len(pool)))))\n        if max(len(ex[k]["ids"]) for k in ["robust","mse","neg"])<=TRAIN_MAX_TOKENS:EXAMPLES.append(ex)\n    if not EXAMPLES:raise RuntimeError("Training prompt too long: increase TRAIN_MAX_TOKENS or shorten feature descriptions")\n    def order_batches():\n        rr=random.Random(SEED)\n        for ep in range(TRAIN_EPOCHS):\n            ids=list(range(len(EXAMPLES)));rr.shuffle(ids)\n            for j in range(0,len(ids),PROMPTS_PER_STEP):yield ids[j:j+PROMPTS_PER_STEP]\n    N_UPDATES=sum(1 for _ in order_batches());ADAPTER_ROOT=Path(OUTPUT_DIR)/"adapters"\n    def ensure_adapter(name):\n        m=HF["model"]\n        cfg=LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.,task_type="CAUSAL_LM",target_modules=["q_proj","k_proj","v_proj","o_proj"])\n        torch.manual_seed(SEED)\n        if not hasattr(m,"peft_config"):\n            m=prepare_model_for_kbit_training(m,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={"use_reentrant":False})\n            m=get_peft_model(m,cfg,adapter_name=name)\n        elif name not in m.peft_config:m.add_adapter(name,cfg)\n        HF["model"]=m;m.set_adapter(name)\n        pars=[]\n        for n,p in m.named_parameters():\n            on="lora_" in n and f".{name}." in n;p.requires_grad_(on)\n            if on:pars.append(p)\n        return m,pars\n    def token_logp(m,ex):\n        x=torch.tensor([ex["ids"]],device=m.device);n=ex["n_ans"]\n        try:o=m(input_ids=x,logits_to_keep=n+1)\n        except TypeError:o=m(input_ids=x)\n        logits=o.logits[0,-n-1:-1].float()\n        return torch.log_softmax(logits,dim=-1).gather(-1,x[0,-n:,None]).squeeze(-1)\n    def amp():return torch.autocast("cuda",dtype=torch.float16)\n    def rollout_reward(text):\n        vals=[];seen=set()\n        for p in parse_structures(text)[:TARGETS_PER_ANSWER]:\n            try:\n                terms=canonical_structure(p["terms"]);key=json.dumps(terms)\n                if key in seen:vals.append(-.5);continue\n                seen.add(key);s=score_structure(terms)\n                vals.append(float(np.clip(s["robust"]/REWARD_SCALE,-1,1))-(.1 if key in {r["key"] for r in POOL} else 0))\n            except Exception:vals.append(-1.)\n        vals += [-1.]*(TARGETS_PER_ANSWER-len(vals))\n        return float(np.mean(vals))\n    for objective in OBJECTIVES:\n        if objective=="dpo" and not NEG:\n            TRAINING_NOTES.append("dpo skipped: no distinct lower-scoring preference candidate; SFT is unaffected.")\n            continue\n        adapter_name=FAMILY+"__"+objective\n        adapter_path=ADAPTER_ROOT/adapter_name;log_path=Path(OUTPUT_DIR)/f"train_{objective}.csv"\n        if RESUME and (adapter_path/"adapter_config.json").exists():\n            from peft import PeftModel\n            m=HF["model"]\n            if hasattr(m,"peft_config"):\n                if adapter_name not in m.peft_config:m.load_adapter(str(adapter_path),adapter_name=adapter_name,is_trainable=False)\n            else:m=PeftModel.from_pretrained(m,str(adapter_path),adapter_name=adapter_name,is_trainable=False)\n            HF["model"]=m;TRAINED.append(objective)\n            TRAIN_LOGS[objective]=pd.read_csv(log_path).to_dict("records") if log_path.exists() else []\n            print("Loaded completed adapter",objective);continue\n        m,params=ensure_adapter(adapter_name)\n        opt=torch.optim.AdamW(params,lr=TRAIN_LR,weight_decay=0.)\n        sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:.5*(1+math.cos(math.pi*min(s/N_UPDATES,1))))\n        scaler=torch.amp.GradScaler("cuda",init_scale=1024)\n        logs=[]\n        for step,batch in enumerate(order_batches(),1):\n            opt.zero_grad(set_to_none=True);values=[];rollouts=[]\n            if objective=="grpo":\n                for i in batch:\n                    texts,prompt_ids,gens=generate_structures(EXAMPLES[i]["prompt"],adapter_name,GRPO_K,True)\n                    rewards=np.array([rollout_reward(t) for t in texts]);adv=(rewards-rewards.mean())/(rewards.std()+1e-6)\n                    stops=HF["model"].generation_config.eos_token_id;stops=set(stops if isinstance(stops,list) else [stops])\n                    for a,g in zip(adv,gens):\n                        ids=g.tolist()\n                        for j,t in enumerate(ids):\n                            if t in stops:ids=ids[:j+1];break\n                        if ids:rollouts.append((float(a),dict(ids=prompt_ids.tolist()+ids,n_ans=len(ids))))\n                    values.append(float(rewards.mean()))\n            m=HF["model"];m.set_adapter(adapter_name);m.train();m.config.use_cache=False\n            if objective in ["ei_mse","ei_robust"]:\n                rule="mse" if objective=="ei_mse" else "robust"\n                for i in batch:\n                    with amp():loss=-token_logp(m,EXAMPLES[i][rule]).mean()\n                    scaler.scale(loss/len(batch)).backward();values.append(float(loss.detach()))\n            elif objective=="dpo":\n                for i in batch:\n                    ch,rj=EXAMPLES[i]["robust"],EXAMPLES[i]["neg"]\n                    with torch.no_grad(),amp(),m.disable_adapter():\n                        ref=token_logp(m,ch).sum()-token_logp(m,rj).sum()\n                    with amp():\n                        margin=DPO_BETA*(token_logp(m,ch).sum()-token_logp(m,rj).sum()-ref)\n                        loss=-torch.nn.functional.logsigmoid(margin)\n                    scaler.scale(loss/len(batch)).backward();values.append(float(loss.detach()))\n            else:\n                if not rollouts:raise RuntimeError("No usable GRPO completion")\n                for a,ex in rollouts:\n                    with torch.no_grad(),amp(),m.disable_adapter():ref=token_logp(m,ex)\n                    with amp():\n                        pol=token_logp(m,ex);d=ref-pol;kl=(torch.exp(d)-d-1).mean()\n                        loss=(-a*pol.mean()+GRPO_BETA*kl)/len(rollouts)\n                    scaler.scale(loss).backward()\n            scaler.unscale_(opt);gn=float(torch.nn.utils.clip_grad_norm_(params,1.))\n            before=scaler.get_scale();scaler.step(opt);scaler.update();skipped=scaler.get_scale()<before or not np.isfinite(gn)\n            if not skipped:sched.step()\n            m.eval();m.config.use_cache=True\n            logs.append(dict(objective=objective,step=step,value=float(np.mean(values)),metric="reward" if objective=="grpo" else "loss",grad_norm=gn,skipped=skipped))\n            print(objective,step,"/",N_UPDATES,"value",round(float(np.mean(values)),4),"skipped",skipped)\n        if not logs or all(x["skipped"] for x in logs):\n            TRAINING_NOTES.append(f"{objective}: all optimizer updates skipped; not treated as trained.")\n        else:\n            m.save_pretrained(str(ADAPTER_ROOT),selected_adapters=[adapter_name]);TRAINED.append(objective)\n        TRAIN_LOGS[objective]=logs;pd.DataFrame(logs).to_csv(log_path,index=False)\nPath(OUTPUT_DIR,"training_status.json").write_text(json.dumps({"trained":TRAINED,"notes":TRAINING_NOTES,"pool":DIAGNOSTICS,"completed_updates":{k:sum(not r["skipped"] for r in v) for k,v in TRAIN_LOGS.items()}},indent=2))\n', 'selection': '# Both post arms start from the SAME pretraining context. Base never gets adapter-generated feedback.\nif USE_LLM:\n    if RUN_BASE_CONTROL:run_search("post_base",PRETRAIN_CONTEXT,POST_ROUNDS)\n    for objective in TRAINED:run_search(f"post_{objective}",PRETRAIN_CONTEXT,POST_ROUNDS,FAMILY+"__"+objective)\n# Everything below is after structure search and training; VAL never returns to the generator.\nvalid={}\nfor r in RECORDS:\n    if r["status"]=="ok":\n        if r["key"] not in valid:valid[r["key"]]=dict(r,sources=[r["source"]])\n        else:valid[r["key"]]["sources"]=sorted(set(valid[r["key"]]["sources"]+[r["source"]]))\nbase_val=loss_of(BASE_MODEL,IDX["val"])\nVALIDATION=[]\nfor r in valid.values():\n    try:\n        model=fitted(r["terms"]);loss=loss_of(model,IDX["val"])\n        ablated=fit_equation(without_news(r["terms"]),IDX["train"])\n        aloss=loss_of(ablated,IDX["val"])\n        VALIDATION.append(dict(r,val_gain=ratio_gain(loss,base_val),val_mse=float(loss.mean()),\n                              val_news_gain=ratio_gain(loss,aloss) if r["uses_news"] else None,\n                              val_imputed_share=input_imputation_share(model,IDX["val"])))\n    except Exception as exc:print("Validation evaluation failed",r["terms"],str(exc))\nif not VALIDATION:raise RuntimeError("No evaluable equations")\nfor r in VALIDATION:r["eligible"]=(r["robust"]>0 and r["cv_consistency"]>=MIN_CV_CONSISTENCY and r["val_gain"]>0)\nranked=sorted(VALIDATION,key=lambda r:(-int(r["eligible"]),-r["val_gain"],-r["robust"],len(r["terms"]),r["complexity"]))\n# Avoid filling the final list with almost identical numerical predictions.\nSELECTED=[];preds=[]\nfor r in ranked:\n    p=predict_equation(fitted(r["terms"]),IDX["val"])\n    equivalent=False\n    for q in preds:\n        if np.std(p)>1e-10 and np.std(q)>1e-10:\n            c=np.corrcoef(p,q)[0,1]\n            # Same sign AND approximately same predictions; scale differences matter for regression.\n            rel=np.mean((p-q)**2)/max(np.mean(q*q),1e-12)\n            if c>EQUIVALENCE_RHO and rel<.01:equivalent=True;break\n    if equivalent:continue\n    SELECTED.append(dict(r,equation_id=f"E{len(SELECTED)+1:02d}"));preds.append(p)\n    if len(SELECTED)>=TOP_FINAL:break\npd.DataFrame(VALIDATION).to_csv(Path(OUTPUT_DIR)/"validation_candidates.csv",index=False)\n# Store exact structures and coefficients BEFORE accessing TEST outcomes.\nPREREG=[dict(equation_id=r["equation_id"],terms=r["terms"],model=fitted(r["terms"]),\n             ablation_model=fit_equation(without_news(r["terms"]),IDX["train"]),sources=r["sources"],\n             cv_gain=r["cv_gain"],cv_consistency=r["cv_consistency"],robust=r["robust"],val_gain=r["val_gain"],\n             hypothesis=r["hypothesis"],uses_news=r["uses_news"]) for r in SELECTED]\npretext=json.dumps({"run_hash":RUN_HASH,"baseline":BASE_MODEL,"equations":PREREG},sort_keys=True,indent=2)\nprepath=Path(OUTPUT_DIR)/"preregistered_equations.json"\nif prepath.exists() and prepath.read_text()!=pretext:raise RuntimeError("Frozen equations changed; refusing to overwrite test registration")\nprepath.write_text(pretext)\ncomp=[]\nfor source in sorted({r["source"] for r in RECORDS}):\n    rows=[r for r in RECORDS if r["source"]==source]\n    ok=[r for r in rows if r["status"]=="ok"]\n    vals=[valid_r["val_gain"] for r in ok for valid_r in VALIDATION if valid_r["key"]==r["key"]]\n    comp.append(dict(source=source,proposed=len(rows),new_valid=len(ok),mean_cv_gain=np.mean([r["cv_gain"] for r in ok]) if ok else np.nan,\n                     mean_val_gain=float(np.mean(vals)) if vals else np.nan,positive_val=int(sum(v>0 for v in vals))))\npd.DataFrame(comp).to_csv(Path(OUTPUT_DIR)/"generator_comparison.csv",index=False)\ndisplay(pd.DataFrame(SELECTED)[["equation_id","terms","sources","cv_gain","val_gain","cv_consistency"]])\nprint("Frozen",len(PREREG),"equations. Coefficients remain fitted on TRAIN.")\n', 'evaluation': 'def nw_summary(x):\n    x=np.asarray(x,float)\n    if not np.isfinite(x).all():raise ValueError("NW expects the fixed finite daily series")\n    T=len(x)\n    if T<20:return np.nan,1.\n    e=x-x.mean();v=float(e@e/T);L=min(NW_LAGS,T-1)\n    for l in range(1,L+1):v+=2*(1-l/(L+1))*float(e[l:]@e[:-l])/T\n    if v<=0:return np.nan,1.\n    t=float(x.mean()/np.sqrt(v/T));return t,float(norm.sf(t))\n\ndef block_indices(n,draws,seed):\n    r=np.random.default_rng(seed);L=min(BOOT_BLOCK,n);k=int(np.ceil(n/L))\n    return (r.integers(0,n-L+1,size=(draws,k,1))+np.arange(L)).reshape(draws,-1)[:,:n]\n\ndef paired_evidence(loss,reference):\n    if not loss.index.equals(reference.index):raise AssertionError("Unpaired dates")\n    a,b=loss.to_numpy(float),reference.to_numpy(float);idx=block_indices(len(a),BOOTSTRAPS,SEED+700)\n    denominator=b[idx].mean(axis=1)\n    bg=1-a[idx].mean(axis=1)/np.maximum(denominator,1e-30)\n    lo,hi=np.quantile(bg,[.025,.975]);t,p=nw_summary(b-a)\n    return dict(gain=ratio_gain(loss,reference),ci_low=float(lo),ci_high=float(hi),nw_t=t,p_raw=p)\n\ndef holm_adjust(p):\n    p=np.asarray(p,float);order=np.argsort(p);out=np.empty(len(p))\n    out[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]))\n    return out\n\ndef coefficient_bootstrap(model):\n    # Fast date-block bootstrap using daily sufficient statistics. No stock-row iid bootstrap.\n    ix=IDX["train"];X=matrix_terms(model["terms"],ix);X=np.where(np.isfinite(X),X,np.asarray(model["medians"]))\n    mu=np.asarray(model["means"]);sd=np.asarray(model["scales"])\n    Z=np.column_stack([np.ones(len(ix)),(X-mu)/sd]);y=Y[ix]\n    unique,day=np.unique(DAY[ix],return_inverse=True);D=len(unique);q=Z.shape[1]\n    cnt=np.bincount(day);w=1/cnt[day]\n    G=np.zeros((D,q,q));h=np.zeros((D,q))\n    for a in range(q):\n        h[:,a]=np.bincount(day,weights=w*Z[:,a]*y,minlength=D)\n        for b in range(a,q):\n            G[:,a,b]=np.bincount(day,weights=w*Z[:,a]*Z[:,b],minlength=D);G[:,b,a]=G[:,a,b]\n    penalty=np.eye(q)*RIDGE_LAMBDA;penalty[0,0]=0\n    indices=block_indices(D,COEF_BOOTSTRAPS,SEED+701);betas=[]\n    for start in range(0,COEF_BOOTSTRAPS,32):\n        take=indices[start:start+32];gg=G[take].mean(axis=1);hh=h[take].mean(axis=1)\n        b=np.linalg.solve(gg+penalty,hh[...,None])[...,0]\n        raw=b[:,1:]/sd\n        betas.append(np.column_stack([b[:,0]-raw@mu,raw]))\n    bs=np.vstack(betas);point=np.r_[model["intercept"],model["coef"]]\n    lo,hi=np.quantile(bs,[.025,.975],axis=0)\n    return [dict(term=term,coefficient=float(v),ci_low=float(l),ci_high=float(u),\n                 bootstrap_same_sign=float(np.mean(np.sign(bs[:,j])==np.sign(v))),\n                 interval_scope="conditional on selected structure and fitted preprocessing")\n            for j,(term,v,l,u) in enumerate(zip(["intercept"]+model["terms"],point,lo,hi))]\n\nlock_hash=hashlib.sha256(pretext.encode()).hexdigest();lock_path=Path(OUTPUT_DIR)/"locked_evaluation.json"\nif RESUME and lock_path.exists():\n    locked=json.loads(lock_path.read_text())\n    if locked["lock_hash"]!=lock_hash:raise RuntimeError("Test lock mismatch")\n    TEST_ROWS=locked["test_rows"];COEF_ROWS=locked["coefficient_rows"]\n    print("Reusing frozen test evaluation; no rescoring.")\nelse:\n    test=IDX["test"];base_loss=loss_of(BASE_MODEL,test);zero_loss=daily_mse(np.zeros(len(test)),test)\n    TEST_ROWS=[];COEF_ROWS=[]\n    for p in PREREG:\n        loss=loss_of(p["model"],test);stats=paired_evidence(loss,base_loss)\n        row=dict(equation_id=p["equation_id"],test_mse=float(loss.mean()),test_rmse=float(np.sqrt(loss.mean())),\n                 baseline_mse=float(base_loss.mean()),r2_zero=ratio_gain(loss,zero_loss),\n                 gain_vs_price=stats["gain"],gain_ci_low=stats["ci_low"],gain_ci_high=stats["ci_high"],\n                 gain_nw_t=stats["nw_t"],p_price_raw=stats["p_raw"],n_test_days=len(loss),n_test_rows=len(test),\n                 test_imputed_share=input_imputation_share(p["model"],test))\n        yearly=pd.DataFrame({"eq":loss,"baseline":base_loss}).groupby(loss.index.year).mean()\n        row["yearly_gain_vs_price"]={str(y):float(1-r["eq"]/r["baseline"]) for y,r in yearly.iterrows()}\n        if p["uses_news"]:\n            abloss=loss_of(p["ablation_model"],test);news_stats=paired_evidence(loss,abloss)\n            row.update(news_gain=news_stats["gain"],news_ci_low=news_stats["ci_low"],news_ci_high=news_stats["ci_high"],p_news_raw=news_stats["p_raw"])\n        else:row.update(news_gain=None,news_ci_low=None,news_ci_high=None,p_news_raw=None)\n        TEST_ROWS.append(row)\n        for r in coefficient_bootstrap(p["model"]):COEF_ROWS.append(dict(equation_id=p["equation_id"],**r))\n    # One family includes ALL price-benchmark tests AND all news-ablation tests, not separate cherry-picked families.\n    tests=[(i,"price",r["p_price_raw"]) for i,r in enumerate(TEST_ROWS)]\n    tests += [(i,"news",r["p_news_raw"]) for i,r in enumerate(TEST_ROWS) if r["p_news_raw"] is not None]\n    for (i,kind,p),q in zip(tests,holm_adjust([x[2] for x in tests])):TEST_ROWS[i][f"p_{kind}_holm"]=float(q)\n    # Strict JSON: nonfinite statistics represented as null, never fabricated values.\n    test_clean=json.loads(pd.DataFrame(TEST_ROWS).to_json(orient="records",double_precision=15))\n    coef_clean=json.loads(pd.DataFrame(COEF_ROWS).to_json(orient="records",double_precision=15))\n    lock_path.write_text(json.dumps({"lock_hash":lock_hash,"test_rows":test_clean,"coefficient_rows":coef_clean},indent=2))\n    TEST_ROWS=test_clean;COEF_ROWS=coef_clean\n\nTEST_FRAME=pd.DataFrame(TEST_ROWS)\nRESULT=pd.DataFrame([{k:p[k] for k in ["equation_id","terms","sources","cv_gain","cv_consistency","robust","val_gain","hypothesis","uses_news"]} for p in PREREG]).merge(TEST_FRAME,on="equation_id")\ndef confidence_label(r):\n    if DEMO_MODE:return "SYNTHETIC demonstration only"\n    if r.n_test_days<MIN_TEST_DAYS or r.test_imputed_share>MAX_TEST_IMPUTED_SHARE:return "Insufficient sample or input coverage"\n    supported=(r.cv_consistency>=MIN_CV_CONSISTENCY and r.val_gain>0 and r.gain_vs_price>0 and r.gain_ci_low>0 and r.p_price_holm<.05)\n    if supported:return "Independent out-of-sample support; external replication needed" if HOLDOUT_IS_FRESH else "Historical out-of-sample support; not independent confirmation"\n    if r.gain_vs_price>0:return "Prediction error improved; insufficient evidence"\n    return "Did not outperform the family-specific baseline"\nRESULT["confidence"]=RESULT.apply(confidence_label,axis=1)\ndef news_label(r):\n    if not r.uses_news:return "no news terms"\n    if DEMO_MODE:return "synthetic only"\n    return "Incremental predictive evidence" if r.news_gain>0 and r.news_ci_low>0 and r.p_news_holm<.05 else "Insufficient evidence of incremental news value"\nRESULT["news_evidence"]=RESULT.apply(news_label,axis=1)\n', 'report': 'def numeric_rhs(model):\n    text=f\'{model["intercept"]:.10g}\'\n    for c,t in zip(model["coef"],model["terms"]):text+=f" {\'+\' if c>=0 else \'-\'} {abs(c):.10g} * ({t})"\n    return text\n\ndef latex_term(expr):\n    def fmt(n):\n        if isinstance(n,ast.Name):return r"\\mathrm{"+n.id.replace("_",r"\\_")+"}"\n        if isinstance(n,ast.Constant):return str(n.value)\n        if isinstance(n,ast.UnaryOp):return ("-" if isinstance(n.op,ast.USub) else "+")+fmt(n.operand)\n        if isinstance(n,ast.BinOp):\n            a,b=fmt(n.left),fmt(n.right)\n            if isinstance(n.op,ast.Div):return r"\\frac{"+a+"}{"+b+"}"\n            op="+" if isinstance(n.op,ast.Add) else ("-" if isinstance(n.op,ast.Sub) else r"\\cdot")\n            return r"\\left("+a+op+b+r"\\right)"\n        return r"\\operatorname{"+n.func.id.replace("_",r"\\_")+"}"+r"\\left("+", ".join(fmt(a) for a in n.args)+r"\\right)"\n    return fmt(ast.parse(expr,mode="eval").body)\n\ndef equation_latex(model):\n    rhs=f\'{model["intercept"]:.6g}\'\n    for c,t in zip(model["coef"],model["terms"]):\n        rhs+=f" {\'+\' if c>=0 else \'-\'} {abs(c):.6g}"+r"\\,"+latex_term(t)\n    return r"\\widehat{y}_{i,t}^{("+str(HORIZON)+r")}\\,[\\mathrm{bps}] = "+rhs\n\ndef marginal_relations(model):\n    # Exact derivative for simple raw-variable main effects and pairwise products only.\n    relations=[]\n    for feature in sorted({f for t in model["terms"] for f in term_features(t) if f.startswith("world_")}):\n        pieces=[];complete=True\n        for term,c in zip(model["terms"],model["coef"]):\n            if feature not in term_features(term):continue\n            n=ast.parse(term,mode="eval").body\n            if term==feature:pieces.append(f"({c:.8g})")\n            elif isinstance(n,ast.BinOp) and isinstance(n.op,ast.Mult) and isinstance(n.left,ast.Name) and isinstance(n.right,ast.Name):\n                if n.left.id==feature and n.right.id==feature:pieces.append(f"({2*c:.8g})*{feature}")\n                else:pieces.append(f"({c:.8g})*{n.right.id if n.left.id==feature else n.left.id}")\n            else:complete=False\n        if complete and pieces:relations.append(f"d(y_hat_bps)/d({feature}) = "+" + ".join(pieces))\n    return relations\n\n'}


In [ ]:

# Stage A: search/train/select ALL registered families before any test evaluation.
for FAMILY in LAW_FAMILIES:
    SPEC=SPECS[FAMILY];g=dict(globals());g["SPEC"]={k:v for k,v in SPEC.items() if k!="target"}
    if SPEC["level"]=="date":
        data=MARKET.copy();data[TARGET]=SPEC["target"].reindex(CAL).to_numpy()
        features=[f for f in STOCK_FEATURES if not f.startswith("world_")]+MARKET_WIDE+["past_market_abs","past_dispersion"]
        g["MIN_NAMES"]=1
    else:
        data=STOCK_DF.copy();data[TARGET]=SPEC["target"].reindex(columns=p.columns).to_numpy()[ROW_D,ROW_N]
        features=STOCK_FEATURES
    if data.loc[data["split"]=="train",TARGET].notna().sum()<120:
        SKIPPED.append({"family":FAMILY,"reason":"insufficient target observations"});continue
    g.update(df=data,FEATURES=list(features),ALL_FEATURES=list(features),
             PRICE_BASELINE_FEATURES=SPEC["base"],OUTPUT_DIR=str(Path(ROOT_OUTPUT)/FAMILY),
             SPLIT_MASK={k:(data["split"]==k).to_numpy() for k in ["train","val","test"]})
    Path(g["OUTPUT_DIR"]).mkdir(parents=True,exist_ok=True)
    print("\nRELATION",FAMILY,SPEC["lhs"],SPEC["unit"])
    for name in ["engine","splits","regression","search","llm","training","selection"]:
        exec(compile(MODULES[name],name,"exec"),g)
    STATES.append(g)
if not STATES:raise RuntimeError("No family has sufficient labels")
registration={"families":LAW_FAMILIES,"skipped":SKIPPED,"fresh":HOLDOUT_IS_FRESH,"models":[{"family":g["FAMILY"],"spec":g["SPEC"],"run_hash":g["RUN_HASH"],"equations":g["PREREG"]} for g in STATES]}
regtext=json.dumps(registration,sort_keys=True,indent=2)
rp=Path(ROOT_OUTPUT)/"ALL_PREREGISTERED.json"
if rp.exists() and rp.read_text()!=regtext:raise RuntimeError("All-family registration changed; select a new RUN_NAME; historical data remain exploratory")
rp.write_text(regtext)
print("All structures frozen. Starting one final evaluation per family.")


In [ ]:

ALL_ROWS=[];ALL_COEFS=[]
for g in STATES:
    exec(compile(MODULES["evaluation"],"evaluation","exec"),g)
    exec(compile(MODULES["report"],"format_functions","exec"),g)
    models={r["equation_id"]:r for r in g["PREREG"]};spec=g["SPEC"]
    for r in g["RESULT"].to_dict("records"):
        local_id=r["equation_id"];model=models[local_id]["model"]
        r.update(family=g["FAMILY"],law_type=spec["label"],unit=spec["unit"],sample_level=spec["level"],definition=spec["definition"],equation_id=g["FAMILY"]+"/"+local_id,
                 formula=spec["lhs"]+" = "+g["numeric_rhs"](model),observed_relation=spec["lhs"].replace("E[","observed[")+" = "+g["numeric_rhs"](model)+" + residual",
                 imputation=dict(zip(model["terms"],model["medians"])),trained_objectives=g["TRAINED"],training_notes=g["TRAINING_NOTES"],holdout_status="fresh_user_declared" if HOLDOUT_IS_FRESH else "previously_seen_exploratory")
        r["gain_vs_baseline"]=r.pop("gain_vs_price");r["yearly_gain_vs_baseline"]=r.pop("yearly_gain_vs_price")
        ALL_ROWS.append(r)
    for c in g["COEF_ROWS"]:ALL_COEFS.append(dict(c,equation_id=g["FAMILY"]+"/"+c["equation_id"],unit=spec["unit"]))
    bundle={"spec":spec,"equations":g["PREREG"],"baseline":g["BASE_MODEL"],"news_clip_bounds":NEWS_CLIP_BOUNDS,"news_cap_fitted_before":NEWS_CAP_END,"news_lag":NEWS_LAG,"feature_docs":{f:FEATURE_DOCS.get(f,f) for f in g["FEATURES"]}}
    Path(g["OUTPUT_DIR"],"model_bundle.json").write_text(json.dumps(bundle,ensure_ascii=False,indent=2))
# ONE multiplicity family for all outcomes, models and news incremental tests.
tests=[(i,"baseline",r["p_price_raw"]) for i,r in enumerate(ALL_ROWS)]
tests += [(i,"news",r["p_news_raw"]) for i,r in enumerate(ALL_ROWS) if r["uses_news"]]
for (i,kind,_),q in zip(tests,STATES[0]["holm_adjust"]([v if v is not None and np.isfinite(v) else 1. for _,_,v in tests])):
    ALL_ROWS[i]["p_"+kind+"_holm_global"]=float(q)
for r in ALL_ROWS:
    sufficient=r["n_test_days"]>=MIN_TEST_DAYS and r["test_imputed_share"]<=MAX_TEST_IMPUTED_SHARE
    passed=sufficient and r["robust"]>0 and r["cv_consistency"]>=MIN_CV_CONSISTENCY and r["val_gain"]>0 and r["gain_ci_low"]>0 and r["p_baseline_holm_global"]<.05
    r["confidence"]="SYNTHETIC ONLY" if DEMO_MODE else ("Insufficient sample or input coverage" if not sufficient else (("Independent out-of-sample support; replication needed" if HOLDOUT_IS_FRESH else "Historical exploratory support; confirmation on new data needed") if passed else "Insufficient evidence for the relationship"))
    r["news_evidence"]="No news terms" if not r["uses_news"] else ("Evidence of incremental news value" if sufficient and r["news_ci_low"]>0 and r["p_news_holm_global"]<.05 else "Insufficient evidence of incremental news value")
    if DEMO_MODE:r["news_evidence"]="SYNTHETIC ONLY"
    for key in ["p_price_holm","p_news_holm"]:r.pop(key,None)
LAWS=pd.DataFrame(ALL_ROWS)
LAWS.to_csv(Path(ROOT_OUTPUT)/"LAWS.csv",index=False,encoding="utf-8-sig")
LAWS.to_json(Path(ROOT_OUTPUT)/"LAWS.json",orient="records",force_ascii=False,indent=2)
pd.DataFrame(ALL_COEFS).to_csv(Path(ROOT_OUTPUT)/"COEFFICIENTS.csv",index=False,encoding="utf-8-sig")
report=["# Multi-family empirical relations", "SYNTHETIC ONLY" if DEMO_MODE else "Historical exploratory analysis; no causal-law claim.",
        "All families were registered before TEST. Confidence uses one global Holm family. Intervals are pointwise date-block intervals, not simultaneous intervals. Coefficient intervals condition on the selected structure and preprocessing.",
        "All coefficients fit TRAIN. Inner CV is purged. VAL selects structures. No test-guided retries. News is lagged, but upstream timestamps/LLM labels and survivorship are not audited. A derivative of a fitted formula is not separately validated as a causal law."]
for r in ALL_ROWS:
    txt=f"## {r['equation_id']} · {r['law_type']}\n\n```text\n{r['formula']}\n```\n\nUnit: {r['unit']}; level: {r['sample_level']}. {r['definition']}\n\nCV gain {r['cv_gain']:.3%}; positive folds {r['cv_consistency']:.0%}; VAL gain {r['val_gain']:.3%}. TEST gain vs own baseline {r['gain_vs_baseline']:.3%}, 95% CI [{r['gain_ci_low']:.3%}, {r['gain_ci_high']:.3%}], global Holm p={r['p_baseline_holm_global']:.4g}. {r['confidence']}. {r['news_evidence']}.\n\nTraining: {r['trained_objectives'] or 'skipped'}; {r['training_notes']}"
    report.append(txt);display(Markdown(txt))
Path(ROOT_OUTPUT,"LAW_REPORT.md").write_text("\n\n".join(report),encoding="utf-8")
print("Saved LAWS.csv, LAWS.json, COEFFICIENTS.csv, LAW_REPORT.md in",ROOT_OUTPUT)
